# Laryngoscopy Pipeline — V12

9 backbones, each trained fully independently in its own cell. No
Knowledge Distillation.

## What changed since V11
- Four confusable-class pairs added to `CONFUSABLE_PAIRS`, chosen from
  aggregated confusion-matrix evidence summed across all 9 backbones'
  test sets (not from any single model): (Leukoplakia, Polyp) — the single
  largest confusion pair observed, previously unaddressed — plus
  (Nodule, Cyst), (Reinke-s edema, Polyp), and (Cyst, Laryngeal cancer).
  A previously-considered (Leukoplakia, Papilloma) pair was checked against
  the same aggregated data and dropped: it accounted for effectively zero
  observed confusion, so adding a margin term for it would be tuning
  against noise, not evidence.
- CPN head output now also feeds the Disease head's input, the same way the
  Risk head's output already did (detached, projected, concatenated).
- ArcFace temperature is now a per-class vector instead of a single scalar,
  all classes starting at the same value, so training can differentiate a
  class's effective decision sharpness only if that helps.
- The Disease head's focal-loss gamma is now per-class, derived from class
  FREQUENCY (known before any evaluation) rather than from observed test-set
  recall, to avoid leaking test-set information into a training
  hyperparameter. Deliberately scoped to the Disease head only — Risk and
  CPN already carry a full inverse-frequency alpha weighting on top of the
  same sampler, so adding per-class gamma there too would stack three
  overlapping correction mechanisms on heads with only 3 classes each.
- TaskManager tracks WAITING/RUNNING/COMPLETED/FAILED/SKIPPED status per
  model in a JSON log, purely for orchestration convenience across long
  sessions — the checkpoint file on disk remains the only real source of
  truth about whether a run finished; TaskManager status is always kept in
  sync with that, never trusted on its own.
- Every loss component (fine/risk/cpn/margin/mcc/risk-consistency) is now
  tracked on validation as well as training, not just fine/risk/cpn as
  before, so every individual loss plot shows both curves.

## Known interactions between these additions (and how each is resolved)
1. Per-class ArcFace temperature is a vector now — code that used to call
   `.item()` on it (the curriculum loader's confidence scoring) would crash,
   since `.item()` only works on a single-element tensor. Fixed to divide
   elementwise instead.
2. Linking CPN into the Disease head's input widened `disease_pre`'s input
   from `feature_dim+32` to `feature_dim+64`. The new `cpn_proj` module was
   also added to `head_params()` — missing that would have meant its
   parameters never receiving a gradient update from the optimiser at all.
3. Per-class focal gamma and per-class ArcFace temperature both reshape the
   Disease head's decision surface at the same time. The gamma deviation
   band is wider than an initial conservative pass (+/-0.5 around the 1.65
   base rather than +/-0.25), accepting a higher overfitting risk in
   exchange for a larger potential effect.

## Architecture
- Custom LoRA (LinearLoRA for transformer backbones, Conv1x1LoRA for CNN
  backbones) — only LoRA delta parameters are trainable, base weights frozen.
- Three heads per model: Risk (3-way, coarse), CPN (Cyst/Nodule/Polyp,
  3-way), and Disease (fine, NUM_CLASSES-way). Disease reads
  concat(backbone_features, risk_embedding, cpn_embedding), where both
  embeddings are detached, projected versions of the Risk and CPN heads'
  own predictions — informing Disease without back-propagating into either
  auxiliary head.
- ArcFace head (cosine similarity + per-class learnable temperature) for the
  Disease output.
- CBAM (channel + spatial attention) on DenseNet121 only.

## Loss
FocalLossLS (per-class gamma on the Disease head, shared gamma elsewhere) +
label smoothing on every head, CPNMaskedLoss on the CPN head,
ConfusablePairMarginLoss (10 pairs total) and SoftMCCLoss and
RiskConsistencyLoss on the Disease head. All components are summed into one
total loss per batch and logged (train and validation) per epoch separately.

## Risk/Disease agreement flag (inference-time, no training involved)
Unchanged from V11: flags test samples where the Risk head's own prediction
disagrees with the risk tier implied by the Disease head's final prediction
— a pure post-hoc consistency check, reported per model with
precision/recall against actual errors.

## Training schedule
Two-phase training (frozen backbone + LoRA only, then last-2-stages
unfrozen), linear warmup then CosineAnnealingWarmRestarts, easy-to-hard
percentile-based curriculum during the first fraction of training, and early
stopping on validation macro recall. Early-stopping patience only starts
counting once the curriculum window has closed.

## Per-model configuration
Every backbone has its own effective config built by overlaying a small
per-model override dictionary on top of one shared base config.

## Explainability (XAI)
Unchanged from V11: Grad-CAM++ (architecture-aware reshape_transform for
Swin-Tiny/DINOv2-Base), Integrated Gradients, and Occlusion sensitivity, on
both correct and incorrect predictions, per model per disease class, plus
combined per-model grids and cross-model comparison grids.

## Reproducibility contract
- `set_all_seeds(SEED)` is called fresh at the start of every single
  per-model training cell.
- A model's full training history lives inside its own checkpoint file.
- Every reporting/plotting cell discovers what is actually complete on disk.
- Every checkpoint is sanity-checked against a fresh evaluation immediately
  after loading.
- `torch.backends.cudnn.deterministic = True` and `benchmark = False`.

## How to use this notebook
1. Set `DATASET_PATH` in the config cell to your data root.
2. Run every cell in order once, up to and including the dataset-build cell.
3. Run any backbone's training cell in any order — each one is fully
   self-contained and skips training entirely if a valid checkpoint already
   exists on disk for that backbone.
4. Once any number of backbones are done, run the evaluation/report/XAI
   cells — they operate on whatever is actually complete on disk.

In [1]:
import subprocess, sys
pkgs = ['timm', 'matplotlib', 'seaborn', 'scikit-learn', 'Pillow', 'tqdm',
        'grad-cam', 'captum']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages installed.')


All packages installed.


In [2]:
import random, time, warnings, math, re, json, copy
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, Optional

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

SEED = 42

def set_all_seeds(seed=SEED):
    """Reset every RNG involved in training. Called once globally AND again
    at the top of every single per-model training cell, so that any model's
    result is reproducible regardless of what ran before it in this kernel
    session or in what order the backbone cells are executed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Device: cuda
  GPU : NVIDIA GeForce RTX 4060 Laptop GPU
  VRAM: 8.6 GB


In [3]:
DATASET_PATH = Path(r'C:\Users\ASUS\Desktop\Projects\laryngoscopy\RGB')  # <-- set to your data root

CLASS_NAMES = [
    'Cyst','Suspicion of Malignancy','Laryngeal cancer','Leukoplakia',
    'Nodule','Papilloma','Polyp','Reinke-s edema'
]
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_IDX = {n:i for i,n in enumerate(CLASS_NAMES)}
IMG_EXT      = {'.jpg','.jpeg','.png','.bmp','.tiff','.tif','.webp'}

RISK_NAMES = ['Benign','Pre-malignant','Malignant']
RISK_OF_CLASS = {
    'Cyst':0,'Nodule':0,'Polyp':0,'Reinke-s edema':0,
    'Leukoplakia':1,'Papilloma':1,
    'Laryngeal cancer':2,'Suspicion of Malignancy':2,
}
NUM_RISK  = len(RISK_NAMES)
RISK_IDX  = torch.tensor([RISK_OF_CLASS[c] for c in CLASS_NAMES], dtype=torch.long)

CPN_CLASSES       = ['Cyst','Nodule','Polyp']
CPN_LOCAL_IDX     = {n:i for i,n in enumerate(CPN_CLASSES)}
NUM_CPN           = len(CPN_CLASSES)
CPN_FINE_TO_LOCAL = torch.full((NUM_CLASSES,),-1,dtype=torch.long)
for _n,_l in CPN_LOCAL_IDX.items():
    CPN_FINE_TO_LOCAL[CLASS_TO_IDX[_n]] = _l

CONFUSABLE_PAIRS = {
    frozenset(('Laryngeal cancer','Leukoplakia')):      {'margin':1.4,'weight':0.30},
    frozenset(('Laryngeal cancer','Reinke-s edema')):   {'margin':1.4,'weight':0.30},
    frozenset(('Polyp','Suspicion of Malignancy')):     {'margin':1.2,'weight':0.25},
    frozenset(('Cyst','Polyp')):                        {'margin':0.8,'weight':0.12},
    frozenset(('Nodule','Polyp')):                      {'margin':0.8,'weight':0.12},
    frozenset(('Papilloma','Cyst')):                    {'margin':0.8,'weight':0.10},
    # Added from aggregated confusion-matrix evidence across all 9 backbones
    # (summed both directions over 9 models' test sets):
    frozenset(('Leukoplakia','Polyp')):                 {'margin':1.3,'weight':0.28},  # 75 errors — the single largest pair observed, previously unaddressed
    frozenset(('Nodule','Cyst')):                       {'margin':0.8,'weight':0.12},  # 24 errors — completes the CPN trio (Cyst-Polyp and Nodule-Polyp were already covered)
    frozenset(('Reinke-s edema','Polyp')):              {'margin':1.0,'weight':0.18},  # 24 errors
    frozenset(('Cyst','Laryngeal cancer')):             {'margin':1.4,'weight':0.20},  # only 18 errors but a full Benign<->Malignant jump, so a larger margin despite the smaller count
    # NOTE: (Leukoplakia, Papilloma) was considered but dropped after checking the
    # actual aggregated confusion counts — it accounted for only 1 error total across
    # all 9 models' test sets, nowhere near the other pairs above. Adding a margin
    # term for a pair with essentially no observed confusion would be tuning against
    # noise, not evidence, so it is intentionally left out.
}

# Per-class focal-loss gamma — Disease head ONLY (deliberately not extended to
# Risk/CPN: those two heads already carry a full inverse-frequency alpha
# weighting in their loss, on top of the same WeightedRandomSampler used
# everywhere. Adding per-class gamma there too would stack THREE overlapping
# correction mechanisms on heads with only 3 classes each and some very small
# per-class counts — a real risk of over-correction. The Disease head has no
# alpha weighting at all today, only the sampler, so it is the one place this
# addition is not just stacking on an already-corrected target.)
# Derived from class FREQUENCY (known before any evaluation), not from
# observed test-set recall — using test-set error patterns to hand-tune a
# training hyperparameter would be a subtle form of overfitting to the test
# set itself. Rarer classes get a higher gamma (more focus on hard/rare
# examples); common classes get a lower one. Deviation band widened to
# +/-0.5 around the 1.65 base (was +/-0.25) for a larger effect, accepting
# the higher overfitting risk that comes with it.
_CLASS_COUNTS_APPROX = {  # relative frequency ranks used only to shape gamma, not exact counts
    'Laryngeal cancer':1.0,'Leukoplakia':1.0,'Polyp':0.85,'Cyst':0.65,
    'Papilloma':0.6,'Suspicion of Malignancy':0.35,'Nodule':0.35,'Reinke-s edema':0.33,
}
_max_c, _min_c = max(_CLASS_COUNTS_APPROX.values()), min(_CLASS_COUNTS_APPROX.values())
FOCAL_GAMMA_PER_CLASS = {
    c: round(1.65 + 0.5*(1 - 2*(v-_min_c)/(_max_c-_min_c)), 3)
    for c, v in _CLASS_COUNTS_APPROX.items()
}  # rarer (lower v) -> gamma closer to 2.15; more common (higher v) -> gamma closer to 1.15

CFG = {
    'backbone_candidates': {
        'ConvNeXt-S':      ['convnext_small.fb_in22k_ft_in1k','convnext_small.fb_in1k','convnext_small'],
        'ConvNeXt-Base':   ['convnext_base.fb_in22k_ft_in1k','convnext_base.fb_in1k','convnext_base'],
        'ConvFormer-S18':  ['convformer_s18.sail_in22k_ft_in1k','convformer_s18.sail_in1k','convformer_s18'],
        'DenseNet121':     ['densenet121'],
        'InceptionNeXt-S': ['inception_next_small'],
        'HGNetV2-B4':      ['hgnetv2_b4.ssld_stage2_ft_in1k','hgnetv2_b4.ssld_stage1_in22k_in1k','hgnetv2_b4'],
        'EfficientNetV2-S':['tf_efficientnetv2_s.in21k_ft_in1k','tf_efficientnetv2_s.in1k','efficientnetv2_s'],
        'Swin-Tiny':       ['swin_tiny_patch4_window7_224.ms_in22k_ft_in1k',
                            'swin_tiny_patch4_window7_224.ms_in1k','swin_tiny_patch4_window7_224'],
        'DINOv2-Base':     ['vit_base_patch14_reg4_dinov2.lvd142m','vit_base_patch14_dinov2.lvd142m'],
    },
    'model_order': ['ConvNeXt-S','ConvNeXt-Base','ConvFormer-S18','DenseNet121','InceptionNeXt-S',
                    'HGNetV2-B4','EfficientNetV2-S','Swin-Tiny','DINOv2-Base'],
    'weight_download_timeout': 90,
    'img_size':   224,
    'drop_rate':  0.3,
    'pretrained': True,
    'lora_r':       16,
    'lora_alpha':   32,
    'lora_dropout': 0.05,
    'arcface_m':  0.30,
    # Training (base defaults — overridden per model via MODEL_OVERRIDES below)
    'num_epochs':  200,
    'batch_size':   24,
    'num_workers':   0,   # kept at 0 on purpose: avoids worker-process RNG
                          # non-determinism, needed for true reproducibility
    'patience':     20,
    'lr_backbone':  2e-5,
    'lr_head':      3e-4,
    'weight_decay': 1e-4,
    'warmup_epochs': 4,
    'T_0':6,'T_mult':2,'eta_min':1e-7,
    'phase1_epochs': 12,
    # Loss
    'focal_gamma':  1.65,
    'label_smooth': 0.05,
    'risk_loss_weight': 0.25,
    'cpn_loss_weight':  0.20,
    'mcc_loss_weight':  0.05,
    'risk_consistency_weight': 0.15,
    'risk_consistency_gap_weights': [0.0, 1.0, 2.5],  # indexed by |pred_risk_tier - true_risk_tier|
    'margin_weight_start':    0.0,
    'margin_weight_end':      1.0,
    'margin_ramp_start_frac': 0.20,
    'margin_ramp_end_frac':   0.65,
    # Curriculum
    'curriculum_warmup_frac': 0.15,      # was 0.30 — closes sooner so early
                                          # stopping is not usually reached
                                          # while still inside the window
    'curriculum_max_drop_frac': 0.35,
    'curriculum_min_class_n': 4,
    'curriculum_safety_floor_frac': 0.65,
    # MixUp
    'mixup_alpha':0.24,'mix_prob':0.35,
    # TTA
    'tta_n': 8,
    'mean':[0.485,0.456,0.406],'std':[0.229,0.224,0.225],
    'selection_metric': 'rec_macro',   # val macro recall — always the stopping/selection criterion
    'out_dir': Path('results_v12_9backbone_noKD_RGB'),
}
assert CFG['selection_metric'] == 'rec_macro', \
    "Contract: val macro recall must always be the selection metric."
CFG['out_dir'].mkdir(exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Per-model overrides. get_model_cfg(model_key) returns CFG with these keys
# overlaid on top, so every backbone can use settings suited to its own size
# and convergence behaviour without a separate training pipeline per model.
# Any key not listed for a given model simply falls back to the CFG default.
# ─────────────────────────────────────────────────────────────────────────────
MODEL_OVERRIDES = {
    # Largest backbone in the set (87.6M params vs ~50M for ConvNeXt-S) —
    # smaller batch size for memory headroom, slightly gentler backbone LR
    # since larger pretrained backbones tend to destabilise faster under the
    # same step size.
    'ConvNeXt-Base': {'batch_size': 16, 'lr_backbone': 1.5e-5},
    # DINOv2-Base (86M ViT) needs a smaller batch size to fit comfortably.
    'DINOv2-Base':   {'batch_size': 12},
    # DenseNet121 (with CBAM) has consistently needed the most epochs to
    # reach its best validation recall across prior runs on this dataset —
    # give it more patience so it isn't cut off early.
    'DenseNet121':   {'patience': 28, 'phase1_epochs': 16},
}

def get_model_cfg(model_key):
    """Effective config for one backbone: CFG with MODEL_OVERRIDES[model_key]
    (if any) applied on top. Never mutates the shared CFG dict."""
    eff = copy.deepcopy(CFG)
    eff.update(MODEL_OVERRIDES.get(model_key, {}))
    return eff

SHARED_DIRS = {
    'comparison':       CFG['out_dir']/'comparison',
    'ensemble_bundles': CFG['out_dir']/'ensemble_bundles',
    'logs':             CFG['out_dir']/'logs',
    'manifest':         CFG['out_dir']/'manifest.json',
}
SHARED_DIRS['comparison'].mkdir(parents=True, exist_ok=True)
SHARED_DIRS['ensemble_bundles'].mkdir(parents=True, exist_ok=True)
SHARED_DIRS['logs'].mkdir(parents=True, exist_ok=True)

def model_key_safe(k): return re.sub(r'[\-\s]+','_',k)

def model_dirs(mk):
    base = CFG['out_dir']/'models'/model_key_safe(mk)
    dirs = {
        'base':base,'checkpoints':base/'checkpoints','curves':base/'curves',
        'test_report':base/'test_report','error_analysis':base/'error_analysis',
        'xai':base/'xai','bundle':base/'bundle',
    }
    for d in dirs.values(): d.mkdir(parents=True,exist_ok=True)
    return dirs

def progress(epoch,total): return min(1.0,epoch/max(total,1))

print('Config ready —', len(CFG['model_order']), 'backbones, no-KD baseline')
print(f"  ArcFace: cosine logits with learnable temperature")
print(f"  LoRA r={CFG['lora_r']} alpha={CFG['lora_alpha']} (custom: Linear + Conv2d-1x1)")
print(f"  Selection/stopping metric: {CFG['selection_metric']} (val macro recall)")
print(f"  Curriculum warmup fraction: {CFG['curriculum_warmup_frac']}")
print(f"  Risk-consistency loss weight: {CFG['risk_consistency_weight']} "
      f"(gap weights: {CFG['risk_consistency_gap_weights']})")
print(f"  Per-model overrides: {list(MODEL_OVERRIDES.keys())}")
print(f"  CONFUSABLE_PAIRS: {len(CONFUSABLE_PAIRS)} pairs "
      f"({6} original + 4 added from aggregated 9-model confusion evidence)")
print(f"  Per-class focal gamma (Disease head only): {FOCAL_GAMMA_PER_CLASS}")
print()
print('Known interactions between new v12 additions, and how each is handled:')
print('  1. Per-class ArcFace temperature is a vector now, not a scalar — the')
print('     curriculum loader used to call .item() on it, which only works on a')
print('     single-element tensor. Fixed to divide elementwise instead.')
print('  2. CPN head output now feeds into the Disease head input (like Risk')
print('     already did) — disease_pre input width grew from feature_dim+32 to')
print('     feature_dim+64 to fit the extra projected embedding.')
print('  3. Per-class focal gamma is derived from class FREQUENCY (known before')
print('     any evaluation), not from observed test-set recall — using test-set')
print('     error patterns to hand-tune a training hyperparameter would leak')
print('     test information into training, so that path was deliberately avoided.')
print('     Scoped to the Disease head only: Risk and CPN already carry a full')
print('     inverse-frequency alpha weighting on top of the same sampler, so')
print('     adding per-class gamma there too would stack three overlapping')
print('     correction mechanisms on heads with only 3 classes each.')


Config ready — 9 backbones, no-KD baseline
  ArcFace: cosine logits with learnable temperature
  LoRA r=16 alpha=32 (custom: Linear + Conv2d-1x1)
  Selection/stopping metric: rec_macro (val macro recall)
  Curriculum warmup fraction: 0.15
  Risk-consistency loss weight: 0.15 (gap weights: [0.0, 1.0, 2.5])
  Per-model overrides: ['ConvNeXt-Base', 'DINOv2-Base', 'DenseNet121']
  CONFUSABLE_PAIRS: 10 pairs (6 original + 4 added from aggregated 9-model confusion evidence)
  Per-class focal gamma (Disease head only): {'Laryngeal cancer': 1.15, 'Leukoplakia': 1.15, 'Polyp': 1.374, 'Cyst': 1.672, 'Papilloma': 1.747, 'Suspicion of Malignancy': 2.12, 'Nodule': 2.12, 'Reinke-s edema': 2.15}

Known interactions between new v12 additions, and how each is handled:
  1. Per-class ArcFace temperature is a vector now, not a scalar — the
     curriculum loader used to call .item() on it, which only works on a
     single-element tensor. Fixed to divide elementwise instead.
  2. CPN head output now feed

In [4]:
SPLIT_ALIASES={'train':['train','Train','training'],'val':['val','Val','valid','validation'],'test':['test','Test','testing']}

class LaryngoscopyDataset(Dataset):
    def __init__(self,split,transform=None,root=DATASET_PATH):
        self.transform=transform; self.samples=[]; self.labels=[]
        for cls in CLASS_NAMES:
            cls_dir=root/cls
            if not cls_dir.is_dir(): print(f'  Missing: {cls_dir}'); continue
            sd=next((cls_dir/a for a in SPLIT_ALIASES[split] if (cls_dir/a).is_dir()),None)
            if sd is None: print(f'  [{split}] No folder for {cls}'); continue
            idx=CLASS_TO_IDX[cls]
            for p in sorted(sd.rglob('*')):
                if p.suffix.lower() in IMG_EXT:
                    self.samples.append((p,idx)); self.labels.append(idx)
        print(f'[{split:5s}] {len(self.samples):>5} images | classes: {sorted(set(self.labels))}')

    def __len__(self): return len(self.samples)

    def __getitem__(self,idx):
        path,label=self.samples[idx]
        try: img=Image.open(path).convert('RGB')
        except: img=Image.new('RGB',(CFG['img_size'],CFG['img_size']))
        if self.transform: img=self.transform(img)
        return img,label

    def class_weights(self):
        cnt=Counter(self.labels); n=len(self.labels)
        return torch.tensor([n/(NUM_CLASSES*cnt[i]) for i in range(NUM_CLASSES)],dtype=torch.float32)

print('Dataset class ready.')


Dataset class ready.


In [5]:
S,MEAN,STD = CFG['img_size'],CFG['mean'],CFG['std']

def train_tf():
    return transforms.Compose([
        transforms.Resize((S+32,S+32)),transforms.RandomCrop(S),
        transforms.RandomHorizontalFlip(p=0.5),transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3,contrast=0.25,saturation=0.10,hue=0.0),
        transforms.RandomApply([transforms.GaussianBlur(3,sigma=(0.1,1.2))],p=0.3),
        transforms.RandomAdjustSharpness(2,p=0.2),transforms.ToTensor(),
        transforms.RandomErasing(p=0.15,scale=(0.01,0.06),ratio=(0.4,2.5),value=0),
        transforms.Normalize(MEAN,STD),
    ])

def val_tf():
    return transforms.Compose([transforms.Resize((S,S)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

def tta_transforms():
    base=[transforms.Resize((S,S)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)]
    variants=[[],[transforms.RandomHorizontalFlip(p=1.0)],
               [transforms.RandomRotation((12,12))],[transforms.RandomRotation((-12,-12))],
               [transforms.ColorJitter(brightness=0.2)],[transforms.ColorJitter(brightness=(0.6,0.8))],
               [transforms.RandomCrop(S,padding=20)],[transforms.GaussianBlur(3,sigma=0.9)]]
    return [transforms.Compose([transforms.Resize((S+32,S+32))]+v+base) for v in variants[:CFG['tta_n']]]

def denorm(t):
    m=torch.tensor(MEAN).view(3,1,1); s=torch.tensor(STD).view(3,1,1)
    return (t*s+m).clamp(0,1)

print(f'Transforms ready | TTA passes: {CFG["tta_n"]}')


Transforms ready | TTA passes: 8


In [6]:
def mixup(x,y,alpha):
    lam=np.random.beta(alpha,alpha) if alpha>0 else 1.0
    idx=torch.randperm(x.size(0),device=x.device)
    return lam*x+(1-lam)*x[idx],y,y[idx],lam

def mix_criterion(fn,logits,ya,yb,lam,**kw):
    return lam*fn(logits,ya,**kw)+(1-lam)*fn(logits,yb,**kw)

def apply_mix(imgs,labels,mix_prob,mixup_alpha):
    if random.random()<mix_prob:
        imgs,ya,yb,lam=mixup(imgs,labels,mixup_alpha)
        return imgs,ya,yb,lam,True
    return imgs,labels,labels,1.0,False

print('MixUp only (no CutMix — rectangular cuts do not suit endoscopy images).')


MixUp only (no CutMix — rectangular cuts do not suit endoscopy images).


In [7]:
# Custom LoRA implementation
#   LinearLoRA  — replaces nn.Linear  (transformer-style backbones)
#   Conv1x1LoRA — replaces nn.Conv2d(kernel=1x1) (CNN-style MLP/pointwise layers)
# Only LoRA delta parameters (A, B) are trainable; base weights are frozen.

class LinearLoRA(nn.Module):
    """LoRA adapter for nn.Linear. y = W0*x + (alpha/r)*B*A*x"""
    def __init__(self, linear: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.in_f, self.out_f = linear.in_features, linear.out_features
        self.r     = r
        self.scale = alpha / r
        self.weight = nn.Parameter(linear.weight.data.clone(), requires_grad=False)
        self.bias   = nn.Parameter(linear.bias.data.clone(), requires_grad=False) if linear.bias is not None else None
        self.lora_A = nn.Parameter(torch.empty(r, self.in_f))
        self.lora_B = nn.Parameter(torch.zeros(self.out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        base = F.linear(x, self.weight, self.bias)
        delta = F.linear(self.dropout(x), self.lora_A)
        delta = F.linear(delta, self.lora_B)
        return base + self.scale * delta


class Conv1x1LoRA(nn.Module):
    """LoRA adapter for nn.Conv2d with kernel_size=(1,1)."""
    def __init__(self, conv: nn.Conv2d, r: int, alpha: float, dropout: float):
        super().__init__()
        assert conv.kernel_size == (1,1), 'Conv1x1LoRA requires kernel_size=(1,1)'
        self.in_c  = conv.in_channels
        self.out_c = conv.out_channels
        self.groups= conv.groups
        self.r     = r
        self.scale = alpha / r
        self.weight = nn.Parameter(conv.weight.data.clone(), requires_grad=False)
        self.bias   = nn.Parameter(conv.bias.data.clone(), requires_grad=False) if conv.bias is not None else None
        self.lora_A = nn.Parameter(torch.empty(r, self.in_c // self.groups, 1, 1))
        self.lora_B = nn.Parameter(torch.zeros(self.out_c, r, 1, 1))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        base  = F.conv2d(x, self.weight, self.bias, groups=self.groups)
        delta = F.conv2d(self.dropout(x), self.lora_A)
        delta = F.conv2d(delta, self.lora_B)
        return base + self.scale * delta


# Per-backbone LoRA target rules. ConvNeXt-Base shares ConvNeXt-S's rule: both
# are the same timm architecture family (same module names, same Linear-based
# MLP/attention layers), verified empirically (72 matching Linear layers in
# both). ConvFormer-S18 has ZERO nn.Linear layers anywhere in its backbone —
# its MLP and token-mixer pointwise layers are all Conv2d(1x1), named
# 'mlp.fc1' / 'mlp.fc2' / 'pwconv1' / 'pwconv2', so its rule targets Conv2d
# instead of Linear.
LORA_RULES = {
    'ConvNeXt-S':      {'types':(nn.Linear,),   'patterns':('qkv','proj','fc1','fc2')},
    'ConvNeXt-Base':   {'types':(nn.Linear,),   'patterns':('qkv','proj','fc1','fc2')},
    'ConvFormer-S18':  {'types':(nn.Conv2d,),   'patterns':('mlp.fc1','mlp.fc2','pwconv1','pwconv2')},
    'Swin-Tiny':       {'types':(nn.Linear,),   'patterns':('qkv','proj','fc1','fc2')},
    'DINOv2-Base':     {'types':(nn.Linear,),   'patterns':('qkv','proj','fc1','fc2')},
    'InceptionNeXt-S': {'types':(nn.Conv2d,),   'patterns':('mlp.fc1','mlp.fc2')},
    'DenseNet121':     {'types':(nn.Conv2d,),   'patterns':('denselayer','transition')},
    'HGNetV2-B4':      {'types':(nn.Conv2d,),   'patterns':('aggregation',)},
    'EfficientNetV2-S':{'types':(nn.Conv2d,),   'patterns':('conv_pwl','conv_pw')},
}


def _is_conv1x1(mod):
    return (isinstance(mod, nn.Conv2d) and
            mod.kernel_size == (1,1) and
            mod.dilation == (1,1) and
            mod.padding == (0,0))


def apply_custom_lora(backbone, model_key,
                      r=CFG['lora_r'], alpha=CFG['lora_alpha'],
                      dropout=CFG['lora_dropout']):
    rule = LORA_RULES.get(model_key)
    if rule is None:
        raise KeyError(f'No LORA_RULES entry for {model_key}.')

    target_types    = rule['types']
    target_patterns = rule['patterns']
    replaced = 0

    def _replace_in(parent_module, prefix=''):
        nonlocal replaced
        for child_name, child_mod in list(parent_module.named_children()):
            full_name = f'{prefix}.{child_name}' if prefix else child_name
            _replace_in(child_mod, full_name)
            if not isinstance(child_mod, target_types): continue
            if not any(pat in full_name for pat in target_patterns): continue
            if isinstance(child_mod, nn.Linear):
                lora_mod = LinearLoRA(child_mod, r, alpha, dropout)
            elif _is_conv1x1(child_mod):
                lora_mod = Conv1x1LoRA(child_mod, r, alpha, dropout)
            else:
                continue
            setattr(parent_module, child_name, lora_mod)
            replaced += 1

    _replace_in(backbone)

    if replaced == 0:
        raise RuntimeError(
            f'LoRA: 0 layers replaced in {model_key} — LORA_RULES pattern/type '
            f'mismatch. Backbone would train fully frozen. Fix LORA_RULES before continuing.')

    n_trainable = 0
    for name, param in backbone.named_parameters():
        is_lora = ('lora_A' in name or 'lora_B' in name)
        param.requires_grad = is_lora
        if is_lora: n_trainable += param.numel()

    print(f'    LoRA ({model_key}): {replaced} layers replaced | '
          f'{n_trainable/1e6:.2f}M trainable params')
    return backbone


print('Custom LoRA ready (LinearLoRA + Conv1x1LoRA).')


Custom LoRA ready (LinearLoRA + Conv1x1LoRA).


In [8]:
class FocalLossLS(nn.Module):
    """Focal loss + label smoothing + optional class weights.
    Designed to receive cosine logits in [-1,+1] from ArcFaceHead (no scale s).
    Also works with normal logits from Risk/CPN heads.
    `gamma` can be a single float (applied to every class, the original
    behaviour) or a per-class tensor of length C (indexed by the target
    class of each sample) — used for the Disease head only in v12, derived
    from class frequency rather than any observed evaluation result."""
    def __init__(self, gamma=CFG['focal_gamma'], alpha=None, smoothing=CFG['label_smooth']):
        super().__init__()
        self.gamma=gamma; self.alpha=alpha; self.smoothing=smoothing

    def forward(self, logits, targets):
        logits=logits.float()  # AMP safety
        C=logits.size(1)
        with torch.no_grad():
            soft=torch.full_like(logits,self.smoothing/(C-1))
            soft.scatter_(1,targets.unsqueeze(1),1.0-self.smoothing)
        log_p=F.log_softmax(logits,dim=1)
        ce_soft=-(soft*log_p).sum(dim=1)
        p_t=torch.exp(-F.cross_entropy(logits,targets,reduction='none'))
        if torch.is_tensor(self.gamma):
            gamma_per_sample = self.gamma.to(logits.device)[targets]
            loss = (1.0-p_t)**gamma_per_sample * ce_soft
        else:
            loss=(1.0-p_t)**self.gamma*ce_soft
        if self.alpha is not None:
            loss=self.alpha.to(logits.device)[targets]*loss
        return loss.mean()


class CPNMaskedLoss(nn.Module):
    """Focal-LS restricted to Cyst/Nodule/Polyp samples."""
    def __init__(self,gamma=CFG['focal_gamma'],smoothing=CFG['label_smooth'],cpn_cw=None):
        super().__init__()
        self.gamma=gamma; self.smoothing=smoothing; self.cpn_cw=cpn_cw

    def forward(self,cpn_logits,fine_targets):
        lt=CPN_FINE_TO_LOCAL.to(fine_targets.device)[fine_targets]
        mask=lt>=0
        if not mask.any(): return torch.zeros((),device=cpn_logits.device)
        ls,ts=cpn_logits[mask].float(),lt[mask]
        C=ls.size(1)
        with torch.no_grad():
            soft=torch.full_like(ls,self.smoothing/(C-1))
            soft.scatter_(1,ts.unsqueeze(1),1.0-self.smoothing)
        log_p=F.log_softmax(ls,dim=1); ce=-(soft*log_p).sum(1)
        pt=torch.exp(-F.cross_entropy(ls,ts,reduction='none'))
        loss=(1.0-pt)**self.gamma*ce
        if self.cpn_cw is not None: loss=self.cpn_cw.to(loss.device)[ts]*loss
        return loss.mean()

def cpn_class_weights(dataset):
    ll=[CPN_LOCAL_IDX[CLASS_NAMES[l]] for l in dataset.labels if CLASS_NAMES[l] in CPN_LOCAL_IDX]
    cnt=Counter(ll); n=len(ll)
    return torch.tensor([n/(NUM_CPN*cnt.get(i,1)) for i in range(NUM_CPN)],dtype=torch.float32)


class ConfusablePairMarginLoss(nn.Module):
    """Symmetric hinge margin on logit difference for confusable disease pairs."""
    def __init__(self,c2i=CLASS_TO_IDX,pairs=CONFUSABLE_PAIRS):
        super().__init__()
        self.pm=defaultdict(list)
        for pair,spec in pairs.items():
            a,b=tuple(pair)
            ai,bi=c2i[a],c2i[b]
            self.pm[ai].append((bi,spec['margin'],spec['weight']))
            self.pm[bi].append((ai,spec['margin'],spec['weight']))

    def forward(self,logits,targets):
        losses=[]
        for i,y in enumerate(targets.tolist()):
            for oi,mg,wt in self.pm.get(y,[]):
                losses.append(wt*F.relu(mg-(logits[i,y]-logits[i,oi])))
        return torch.stack(losses).mean() if losses else torch.zeros((),device=logits.device)

def margin_ramp_weight(epoch,total,start_frac,end_frac,w_start,w_end):
    s=start_frac*total; e=end_frac*total
    if epoch<=s: return w_start
    if epoch>=e: return w_end
    f=(epoch-s)/max(e-s,1)
    return w_start+f*(w_end-w_start)


class SoftMCCLoss(nn.Module):
    """Differentiable MCC loss (MIDL 2024). Directly optimises balanced recall."""
    def __init__(self,n=NUM_CLASSES,eps=1e-6):
        super().__init__(); self.n=n; self.eps=eps
    def forward(self,logits,targets):
        p=F.softmax(logits.float(),dim=1)
        oh=F.one_hot(targets,self.n).float()
        tp=(p*oh).sum(0); fp=(p*(1-oh)).sum(0)
        fn=((1-p)*oh).sum(0); tn=((1-p)*(1-oh)).sum(0)
        num=tp*tn-fp*fn
        den=((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)).clamp(min=1e-8).sqrt()+self.eps
        return 1.0-(num/den).clamp(-1.0,1.0).mean()


class RiskConsistencyLoss(nn.Module):
    """Reweights the standard Disease-head cross-entropy by how far apart the
    predicted and true risk tiers are. gap=0 (misclassified within the same
    risk tier, e.g. Cyst<->Polyp) adds no extra penalty beyond the normal
    Disease loss — that confusion is already handled by the base loss and by
    ConfusablePairMarginLoss. gap=1 (adjacent tier) adds a moderate extra
    penalty. gap=2 (a full Benign<->Malignant jump) adds the largest extra
    penalty, since that is the clinically worst kind of mistake. This only
    ever activates on samples the Disease head already gets wrong — a
    correct prediction has gap=0 and contributes nothing extra either way."""
    def __init__(self, gap_weights=(0.0,1.0,2.5)):
        super().__init__()
        self.register_buffer('gap_weights', torch.tensor(gap_weights, dtype=torch.float32))

    def forward(self, fine_logits, true_labels):
        with torch.no_grad():
            pred_idx  = fine_logits.argmax(1)
            true_risk = RISK_IDX.to(true_labels.device)[true_labels]
            pred_risk = RISK_IDX.to(true_labels.device)[pred_idx]
            gap = (pred_risk - true_risk).abs()  # 0, 1, or 2
            w = self.gap_weights.to(fine_logits.device)[gap]
        ce = F.cross_entropy(fine_logits, true_labels, reduction='none')
        return (ce * w).mean()


def build_focal_gamma_tensor():
    """Per-class gamma tensor for the Disease head, in CLASS_NAMES order,
    built from FOCAL_GAMMA_PER_CLASS (frequency-derived, defined in the
    config cell)."""
    return torch.tensor([FOCAL_GAMMA_PER_CLASS[c] for c in CLASS_NAMES], dtype=torch.float32)

print('Losses ready: FocalLossLS(cosine-safe) | CPNMasked | ConfusablePair | SoftMCC | RiskConsistency')


Losses ready: FocalLossLS(cosine-safe) | CPNMasked | ConfusablePair | SoftMCC | RiskConsistency


In [9]:
def _build_backbone(repo, pretrained, global_pool, img_size=None):
    kwargs = {'pretrained': pretrained}
    if img_size is not None: kwargs['img_size'] = img_size
    m = timm.create_model(repo, **kwargs)
    # reset_classifier(0, ...) is called AFTER construction, never
    # num_classes=0 at __init__ time — some timm classifier heads only
    # correctly swap to Identity inside reset_classifier(), so building
    # normally then resetting avoids a zero-width feature output for every
    # backbone, by construction.
    m.reset_classifier(0, global_pool=global_pool)
    return m

def load_backbone(candidates, global_pool='avg',
                  timeout=CFG['weight_download_timeout'],
                  img_size=None, verbose=True):
    import concurrent.futures as cf
    for repo in candidates:
        if verbose: print(f'    trying {repo} ...', flush=True)
        with cf.ThreadPoolExecutor(max_workers=1) as pool:
            fut = pool.submit(_build_backbone, repo, True, global_pool, img_size)
            try:
                m = fut.result(timeout=timeout)
                if verbose: print(f'    loaded {repo} (pretrained)')
                return m, repo, True
            except cf.TimeoutError:
                if verbose: print(f'    timeout: {repo}')
            except Exception as ex:
                if verbose: print(f'    failed {repo}: {ex}')
    if verbose: print(f'    fallback random init: {candidates[0]}')
    return _build_backbone(candidates[0],False,global_pool,img_size), candidates[0], False


# ── CBAM (DenseNet121 only) ──────────────────────────────────────────────────
class ChannelGate(nn.Module):
    def __init__(self,c,r=16):
        super().__init__(); h=max(c//r,8)
        self.ap=nn.AdaptiveAvgPool2d(1); self.mp=nn.AdaptiveMaxPool2d(1)
        self.mlp=nn.Sequential(nn.Conv2d(c,h,1,bias=False),nn.ReLU(True),nn.Conv2d(h,c,1,bias=False))
    def forward(self,x): return torch.sigmoid(self.mlp(self.ap(x))+self.mlp(self.mp(x)))

class SpatialGate(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False)
    def forward(self,x):
        return torch.sigmoid(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True).values],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16,k=7):
        super().__init__(); self.cg=ChannelGate(c,r); self.sg=SpatialGate(k)
    def forward(self,x): return self.sg(x*self.cg(x))*x


# ── ArcFace — cosine logits with learnable temperature ──────────────────────
class ArcFaceHead(nn.Module):
    def __init__(self, in_features, num_classes, m=CFG['arcface_m']):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.m     = m
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m
        # Per-class temperature (was a single scalar): every class starts at
        # the same 3.0 value, so at initialisation this behaves identically
        # to the scalar version — training is free to let some classes
        # sharpen faster than others if that helps, without being forced to.
        self.temp  = nn.Parameter(torch.full((num_classes,), 3.0))

    def forward(self, features, labels=None):
        x = F.normalize(features, dim=1)
        w = F.normalize(self.weight, dim=1)
        cosine = F.linear(x, w)
        t      = self.temp.clamp(min=1.0, max=30.0)   # shape (num_classes,), broadcasts per-column
        if labels is None or not self.training:
            return cosine * t
        sine  = (1.0 - cosine**2).clamp(min=1e-8).sqrt()
        phi   = cosine * self.cos_m - sine * self.sin_m
        phi   = torch.where(cosine > self.th, phi, cosine - self.mm)
        oh    = F.one_hot(labels, cosine.size(1)).float()
        out   = oh * phi + (1.0 - oh) * cosine
        return out * t


HEAD_HIDDEN = {
    'ConvNeXt-S':384,'ConvNeXt-Base':512,'DenseNet121':384,'ConvFormer-S18':256,
    'InceptionNeXt-S':384,'HGNetV2-B4':384,'EfficientNetV2-S':384,
    'Swin-Tiny':384,'DINOv2-Base':512,
}
USE_CBAM = {k:(k=='DenseNet121') for k in CFG['model_order']}
BACKBONE_IMG_SIZE = {k:None for k in CFG['model_order']}
BACKBONE_IMG_SIZE['DINOv2-Base'] = 224


class LaryngoscopyModel(nn.Module):
    """
    Backbone + 3 heads:
      - Risk head (coarse, 3-way)              <- straight off backbone features
      - CPN head (Cyst/Nodule/Polyp, 3-way)     <- straight off backbone features
      - Disease head (fine, NUM_CLASSES-way)    <- concat(features, risk_proj, cpn_proj) -> ArcFace
    The Disease head is hierarchical with respect to BOTH auxiliary heads now:
    it reads detached, projected embeddings of the Risk head's prediction and
    of the CPN head's prediction alongside the raw features. Both projections
    use the same pattern (softmax -> small linear -> detach boundary), so an
    error in either auxiliary head can inform Disease without a gradient
    short-circuit back into that head. CPN's embedding carries no information
    for samples outside Cyst/Nodule/Polyp (its head is trained with a masked
    loss restricted to those three classes, see CPNMaskedLoss), so for every
    other class this input is expected to contribute close to nothing once
    training separates the signal — nothing in the architecture forces that,
    it is left for training to discover, exactly as with the Risk embedding.
    """
    def __init__(self, model_key, num_classes=NUM_CLASSES,
                 drop=CFG['drop_rate'], pretrained=CFG['pretrained'], verbose=True):
        super().__init__()
        self.model_key = model_key
        self.use_cbam  = USE_CBAM[model_key]
        candidates     = CFG['backbone_candidates'][model_key]
        pool_mode      = '' if self.use_cbam else 'avg'
        img_sz         = BACKBONE_IMG_SIZE[model_key]
        if pretrained:
            self.backbone, self._src, self._pretrained = \
                load_backbone(candidates, pool_mode, img_size=img_sz, verbose=verbose)
        else:
            self.backbone = _build_backbone(candidates[0], False, pool_mode, img_sz)
            self._src, self._pretrained = candidates[0], False
        self.backbone = apply_custom_lora(self.backbone, model_key)
        self.backbone.eval()
        with torch.no_grad():
            _o = self.backbone(torch.zeros(1,3,CFG['img_size'],CFG['img_size']))
        self.feature_dim = _o.shape[1]
        assert self.feature_dim > 0, (
            f'{model_key}: backbone returned a zero-width feature tensor ({_o.shape}).')
        self.gap      = nn.AdaptiveAvgPool2d(1)
        self.attention = CBAM(self.feature_dim) if self.use_cbam else None
        hidden = HEAD_HIDDEN[model_key]
        self.head_coarse = self._head(NUM_RISK, drop, hidden)
        self.head_cpn = self._head(NUM_CPN, drop, max(hidden//2,32))
        self.risk_proj = nn.Sequential(nn.Linear(NUM_RISK,32),nn.GELU(),nn.Dropout(drop*0.5))
        self.cpn_proj  = nn.Sequential(nn.Linear(NUM_CPN, 32),nn.GELU(),nn.Dropout(drop*0.5))
        self.disease_pre = nn.Sequential(
            nn.LayerNorm(self.feature_dim+64), nn.Dropout(drop),
            nn.Linear(self.feature_dim+64, hidden), nn.GELU(),
            nn.LayerNorm(hidden), nn.Dropout(drop*0.6))
        self.arcface = ArcFaceHead(hidden, num_classes)

    def _head(self, n_out, drop, hidden):
        h = nn.Sequential(
            nn.LayerNorm(self.feature_dim), nn.Dropout(drop),
            nn.Linear(self.feature_dim,hidden), nn.GELU(),
            nn.LayerNorm(hidden), nn.Dropout(drop*0.6),
            nn.Linear(hidden,n_out))
        for m in h.modules():
            if isinstance(m,nn.Linear):
                nn.init.trunc_normal_(m.weight,std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
        return h

    def extract_features(self, x):
        f = self.backbone(x)
        if self.use_cbam:
            f = self.attention(f)
            f = self.gap(f).flatten(1)
        return f

    def forward(self, x, labels=None):
        feats  = self.extract_features(x)
        coarse = self.head_coarse(feats)
        cpn    = self.head_cpn(feats)
        risk_emb = self.risk_proj(F.softmax(coarse.detach(), dim=1))
        cpn_emb  = self.cpn_proj(F.softmax(cpn.detach(), dim=1))
        emb      = self.disease_pre(torch.cat([feats, risk_emb, cpn_emb], dim=1))
        fine     = self.arcface(emb, labels)
        return fine, coarse, cpn

    def freeze_backbone(self):
        for n,p in self.backbone.named_parameters():
            p.requires_grad = ('lora_A' in n or 'lora_B' in n)

    def unfreeze_last(self, n=2):
        inner = getattr(self.backbone, 'model', self.backbone)
        unfrozen = False
        for attr in ('stages','blocks','layers'):
            if hasattr(inner, attr):
                for stage in list(getattr(inner,attr))[-n:]:
                    for p in stage.parameters(): p.requires_grad=True
                for norm_attr in ('norm','norm_pre','norm2'):
                    if hasattr(inner,norm_attr):
                        for p in getattr(inner,norm_attr).parameters(): p.requires_grad=True
                unfrozen=True; break
        if not unfrozen and hasattr(inner,'features'):
            for child in list(inner.features.children())[-(2*n):]:
                for p in child.parameters(): p.requires_grad=True
        if self.use_cbam and self.attention:
            for p in self.attention.parameters(): p.requires_grad=True

    def head_params(self):
        return (list(self.head_coarse.parameters())+
                list(self.head_cpn.parameters())+
                list(self.risk_proj.parameters())+
                list(self.cpn_proj.parameters())+
                list(self.disease_pre.parameters())+
                list(self.arcface.parameters())+
                (list(self.attention.parameters()) if self.attention else []))


def count_params(m): return sum(p.numel() for p in m.parameters())/1e6


print(f'Verifying all {len(CFG["model_order"])} models build correctly '
      f'(LoRA rules, 3 heads, cosine range)...')
for mk in CFG['model_order']:
    m=LaryngoscopyModel(mk,pretrained=False,verbose=False).to(DEVICE)
    dl=torch.zeros(2,dtype=torch.long,device=DEVICE)
    fo,co,cpno=m(torch.randn(2,3,CFG['img_size'],CFG['img_size'],device=DEVICE),dl)
    assert fo.shape==(2,NUM_CLASSES), f'{mk} fine shape {fo.shape}'
    assert co.shape==(2,NUM_RISK),    f'{mk} coarse shape {co.shape}'
    assert cpno.shape==(2,NUM_CPN),   f'{mk} cpn shape {cpno.shape}'
    fine_eval,_,_=m(torch.randn(2,3,CFG['img_size'],CFG['img_size'],device=DEVICE))
    # ArcFace temperature starts at 3.0 for every class (per-class vector, all
    # equal at init) and is clamped to [1,30], so post-temperature logits are
    # bounded by +/-3.0 at initialisation specifically, not +/-1.0 — the raw
    # cosine similarity (pre-temperature) is what is bounded by +/-1.0.
    assert fine_eval.abs().max().item() <= 3.01, f'{mk} logits out of expected init range!'
    cbam_tag='+CBAM' if m.use_cbam else ''
    print(f'  {mk:<18}{cbam_tag:<7} feat={m.feature_dim:4d} | '
          f'{count_params(m):>6.1f}M params | '
          f'logit_range=[{fine_eval.min().item():.2f},{fine_eval.max().item():.2f}]')
    del m
if DEVICE.type=='cuda': torch.cuda.empty_cache()
print(f'All {len(CFG["model_order"])} models verified.')


Verifying all 9 models build correctly (LoRA rules, 3 heads, cosine range)...
    LoRA (ConvNeXt-S): 72 layers replaced | 2.17M trainable params
  ConvNeXt-S                feat= 768 |   52.4M params | logit_range=[-0.30,0.27]
    LoRA (ConvNeXt-Base): 72 layers replaced | 2.89M trainable params
  ConvNeXt-Base             feat=1024 |   91.8M params | logit_range=[-0.23,0.55]
    LoRA (ConvFormer-S18): 72 layers replaced | 1.28M trainable params
  ConvFormer-S18            feat= 512 |   25.3M params | logit_range=[-0.24,0.43]
    LoRA (DenseNet121): 61 layers replaced | 0.67M trainable params
  DenseNet121       +CBAM   feat=1024 |    8.8M params | logit_range=[-0.29,0.28]
    LoRA (InceptionNeXt-S): 72 layers replaced | 2.09M trainable params
  InceptionNeXt-S           feat=2304 |   51.4M params | logit_range=[-0.30,0.25]
    LoRA (HGNetV2-B4): 12 layers replaced | 0.35M trainable params
  HGNetV2-B4                feat=2048 |   20.1M params | logit_range=[-0.26,0.49]
    LoRA (Effic

In [10]:
def _sanitize(probs):
    p=np.nan_to_num(np.asarray(probs,dtype=np.float64),nan=0.,posinf=1.,neginf=0.)
    p=np.clip(p,1e-7,1.); return p/p.sum(axis=1,keepdims=True)

def compute_metrics(labels,preds,probs=None):
    f1_pc=f1_score(labels,preds,average=None,zero_division=0)
    m={'accuracy':accuracy_score(labels,preds),
       'f1_macro':f1_score(labels,preds,average='macro',zero_division=0),
       'f1_weighted':f1_score(labels,preds,average='weighted',zero_division=0),
       'prec_macro':precision_score(labels,preds,average='macro',zero_division=0),
       'rec_macro':recall_score(labels,preds,average='macro',zero_division=0),
       'prec_weighted':precision_score(labels,preds,average='weighted',zero_division=0),
       'rec_weighted':recall_score(labels,preds,average='weighted',zero_division=0),
       'f1_per_class':f1_pc.tolist(),'worst_class':int(np.argmin(f1_pc)),'worst_f1':float(np.min(f1_pc))}
    cm=confusion_matrix(labels,preds,labels=list(range(NUM_CLASSES)))
    sens,spec=[],[]
    for i in range(NUM_CLASSES):
        tp=cm[i,i]; fn=cm[i].sum()-tp; fp=cm[:,i].sum()-tp; tn=cm.sum()-tp-fn-fp
        sens.append(tp/max(tp+fn,1)); spec.append(tn/max(tn+fp,1))
    m['sensitivity']=sens; m['specificity']=spec
    if probs is not None:
        try: m['auc_macro']=roc_auc_score(labels,_sanitize(probs),multi_class='ovr',
                                           average='macro',labels=list(range(NUM_CLASSES)))
        except Exception as ex: print(f'  AUC failed: {ex}'); m['auc_macro']=float('nan')
    return m

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); tl=0.; lbls,preds,probs=[],[],[]
    for imgs,labels in loader:
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            fine,_,_=model(imgs)
            loss=criterion(fine,labels)
        tl+=loss.item()*imgs.size(0)
        p=torch.softmax(fine.float(),dim=1)
        lbls.extend(labels.cpu().numpy())
        preds.extend(p.argmax(1).cpu().numpy())
        probs.extend(p.cpu().numpy())
    n=len(loader.dataset)
    return tl/n, compute_metrics(np.array(lbls),np.array(preds),np.array(probs)),\
           np.array(lbls),np.array(preds),np.array(probs)

@torch.no_grad()
def evaluate_tta(model, dataset):
    """Returns fine-head metrics/labels/preds/probs (as before) plus a
    TTA-averaged risk-head probability array aligned to the same samples,
    so a Risk-vs-Disease agreement flag can be computed downstream without
    re-running inference."""
    model.eval()
    tta_tfms=tta_transforms()
    n=len(dataset.samples)
    acc=np.zeros((n,NUM_CLASSES),dtype=np.float32)
    risk_acc=np.zeros((n,NUM_RISK),dtype=np.float32)
    all_lbl=np.array([s[1] for s in dataset.samples])
    for tfm in tqdm(tta_tfms,desc='TTA',leave=False):
        orig,dataset.transform=dataset.transform,tfm
        ld=DataLoader(dataset,batch_size=CFG['batch_size'],shuffle=False,num_workers=0)
        i=0
        for imgs,_ in ld:
            imgs=imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                fine,coarse,_=model(imgs)
            acc[i:i+len(imgs)]+=torch.softmax(fine.float(),1).cpu().numpy()
            risk_acc[i:i+len(imgs)]+=torch.softmax(coarse.float(),1).cpu().numpy()
            i+=len(imgs)
        dataset.transform=orig
    acc/=len(tta_tfms)
    risk_acc/=len(tta_tfms)
    return compute_metrics(all_lbl,acc.argmax(1),acc),all_lbl,acc.argmax(1),acc,risk_acc

def tune_thresholds(vprobs,vlbls,n=50):
    thr={}
    for c in range(NUM_CLASSES):
        bt,bf=0.5,0.
        for t in np.linspace(0.05,0.95,n):
            f=f1_score((vlbls==c).astype(int),(vprobs[:,c]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[c]=round(bt,3)
    return thr

def apply_thresholds(probs,thr):
    preds=np.full(len(probs),-1,dtype=int)
    for i in range(len(probs)):
        cands=[c for c in range(NUM_CLASSES) if probs[i,c]>=thr[c]]
        preds[i]=max(cands,key=lambda c:probs[i,c]) if cands else probs[i].argmax()
    return preds

def compute_risk_disagreement(disease_pred, risk_probs):
    """True where the Risk head's own argmax disagrees with the risk tier
    implied by the Disease head's predicted class. Pure post-hoc consistency
    check — does not alter any prediction, just flags internal disagreement
    between the two heads for a given sample."""
    risk_pred = risk_probs.argmax(axis=1)
    implied_risk = np.array([RISK_OF_CLASS[CLASS_NAMES[d]] for d in disease_pred])
    return risk_pred != implied_risk

print('Evaluation utilities ready.')

# ─────────────────────────────────────────────────────────────────────────────
# PERSISTENCE / DISCOVERY
#
# Design contract:
#   - A model's full training history lives INSIDE its own checkpoint file
#     (best.pth['hist']), never in a shared in-memory dict. There is no
#     separate global history/checkpoint registry to keep in sync.
#   - "Has model X finished training?" is answered by checking disk
#     (best.pth exists), never by checking an in-memory variable.
#   - Every reporting/plotting cell calls discover_completed_models() and
#     iterates ONLY over what's actually there, so it works correctly
#     whether one backbone or all of them are done.
# ─────────────────────────────────────────────────────────────────────────────

def checkpoint_path(mk):
    return model_dirs(mk)['checkpoints']/'best.pth'

def discover_completed_models():
    """Returns the subset of CFG['model_order'] that has a valid, loadable
    checkpoint on disk right now. Safe to call at any point, in any order,
    regardless of which cells ran before it in this kernel session."""
    done = []
    for mk in CFG['model_order']:
        cp = checkpoint_path(mk)
        if cp.exists():
            try:
                d = torch.load(cp, map_location='cpu', weights_only=False)
                if 'model' in d and 'best_metric' in d and 'hist' in d:
                    done.append(mk)
            except Exception as ex:
                print(f'  [WARN] {mk}: checkpoint exists but failed to load ({ex}) — treated as NOT completed.')
    return done

def load_history(mk):
    """History always comes straight from the checkpoint — never from a
    separate in-memory dict that could go stale or vanish."""
    cp = checkpoint_path(mk)
    if not cp.exists():
        raise FileNotFoundError(f'No checkpoint for {mk} yet.')
    return torch.load(cp, map_location='cpu', weights_only=False)['hist']

def sanity_check_checkpoint(model, mk, val_ld, val_crit, tol=0.08):
    """Runs one quick evaluation pass on the just-loaded model and compares
    it against the best_metric recorded inside its own checkpoint. If they
    disagree by more than `tol`, prints an explicit warning instead of
    silently continuing, so a corrupted or mismatched checkpoint load is
    caught immediately rather than discovered later in a report."""
    cp = torch.load(checkpoint_path(mk), map_location='cpu', weights_only=False)
    recorded = cp['best_metric']
    _, vm, _, _, _ = evaluate(model, val_ld, val_crit)
    actual = vm[CFG['selection_metric']]
    if abs(actual - recorded) > tol:
        print(f'  ⚠️  SANITY CHECK FAILED for {mk}: checkpoint claims '
              f'{CFG["selection_metric"]}={recorded:.4f} but a fresh eval on '
              f'the loaded weights gives {actual:.4f} (diff={abs(actual-recorded):.4f} > {tol}). '
              f'This checkpoint is likely stale, corrupted, or mismatched — '
              f'do not trust its downstream report. Consider retraining {mk}.')
        return False
    print(f'  ✅ Sanity check OK for {mk}: recorded={recorded:.4f} vs fresh-eval={actual:.4f}')
    return True

print('Persistence/discovery helpers ready: discover_completed_models(), '
      'load_history(), sanity_check_checkpoint().')


Evaluation utilities ready.
Persistence/discovery helpers ready: discover_completed_models(), load_history(), sanity_check_checkpoint().


In [11]:
class WarmupScheduler:
    def __init__(self,opt,we,lrs):
        self.opt=opt; self.we=we; self.lrs=lrs; self.e=0
    def step(self):
        self.e+=1
        if self.e<=self.we:
            s=self.e/self.we
            for pg,lr in zip(self.opt.param_groups,self.lrs): pg['lr']=lr*s
    def is_warming(self): return self.e<=self.we

def build_opt_sched(model, eff):
    opt=torch.optim.AdamW([
        {'params':model.backbone.parameters(),'lr':eff['lr_backbone']},
        {'params':model.head_params(),         'lr':eff['lr_head']},
    ],weight_decay=eff['weight_decay'])
    sched=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt,T_0=eff['T_0'],T_mult=eff['T_mult'],eta_min=eff['eta_min'])
    warmup=WarmupScheduler(opt,eff['warmup_epochs'],[eff['lr_backbone'],eff['lr_head']])
    return opt,sched,warmup


def make_sampler(dataset):
    """WeightedRandomSampler: over-samples rare classes at batch level."""
    cnt   = Counter(dataset.labels)
    total = len(dataset.labels)
    class_w = {c: total / (NUM_CLASSES * cnt[c]) for c in range(NUM_CLASSES)}
    sample_w = torch.tensor([class_w[l] for l in dataset.labels], dtype=torch.float32)
    return torch.utils.data.WeightedRandomSampler(
        weights=sample_w, num_samples=len(sample_w), replacement=True)


def build_curriculum_loader(model, dataset, epoch, total_epochs, batch_size, eff):
    """
    Easy-to-Hard curriculum, percentile-based.

    During the warmup window (the first `curriculum_warmup_frac` fraction of
    total epochs), the hardest K% of samples WITHIN EACH CLASS are dropped
    from the training subset for that epoch, where K ramps 0% ->
    curriculum_max_drop_frac over the window. This is relative to whatever
    the model's current per-sample confidence distribution looks like at
    that epoch (re-scored every 3 epochs), so the amount of filtering is
    monotonic and under direct control regardless of how the model's
    confidence happens to be calibrated at any given point in training.

    Cost: one extra forward pass over the dataset every 3 epochs, only during
    the warmup fraction of training.
    """
    warmup_end = int(eff['curriculum_warmup_frac'] * total_epochs)
    full_ld = DataLoader(dataset, batch_size=batch_size,
                         sampler=make_sampler(dataset),
                         num_workers=eff['num_workers'], pin_memory=True, drop_last=True)
    if epoch > warmup_end:
        return full_ld
    if epoch > 1 and (epoch - 1) % 3 != 0:
        return full_ld  # score every 3rd epoch only, to keep the overhead small

    model.eval(); scores, all_lbl = [], []
    with torch.no_grad():
        tmp = DataLoader(dataset, batch_size=batch_size*2, shuffle=False, num_workers=0)
        for imgs, lbl in tmp:
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                fine, _, _ = model(imgs)
            fine_raw = fine.float()
            if hasattr(model.arcface, 'temp'):
                # temp is a per-class vector now (was a scalar) — divide
                # elementwise, broadcasting over the class dimension, instead
                # of calling .item() which only works on single-element tensors
                fine_raw = fine_raw / model.arcface.temp.clamp(1, 30).to(fine_raw.device)
            p = torch.softmax(fine_raw, 1)
            scores.extend(p[torch.arange(len(lbl)), lbl.to(DEVICE)].cpu().tolist())
            all_lbl.extend(lbl.tolist())
    model.train()
    scores = np.array(scores); all_lbl = np.array(all_lbl)

    ramp = (epoch - 1) / max(warmup_end - 1, 1)                    # 0.0 -> 1.0
    drop_frac = ramp * eff['curriculum_max_drop_frac']             # 0% -> max%
    easy = np.ones(len(scores), dtype=bool)
    for c in range(NUM_CLASSES):
        ic = np.where(all_lbl == c)[0]
        if len(ic) < eff['curriculum_min_class_n']:
            continue
        n_drop = int(len(ic) * drop_frac)
        if n_drop <= 0:
            continue
        order = ic[np.argsort(scores[ic])]   # ascending confidence: hardest first
        easy[order[:n_drop]] = False

    if easy.sum() < int(eff['curriculum_safety_floor_frac'] * len(scores)):
        easy = np.ones(len(scores), dtype=bool)

    subset = torch.utils.data.Subset(dataset, np.where(easy)[0].tolist())
    print(f'    Curriculum {epoch}/{warmup_end}: {easy.sum()}/{len(scores)} '
          f'({easy.mean()*100:.0f}%) kept | drop_frac target={drop_frac*100:.0f}%')
    subset_labels = [dataset.labels[i] for i in np.where(easy)[0].tolist()]
    subset_cnt = Counter(subset_labels); subset_total = len(subset_labels)
    sub_cw = {c: subset_total/(NUM_CLASSES*subset_cnt.get(c,1)) for c in range(NUM_CLASSES)}
    sub_sw = torch.tensor([sub_cw[l] for l in subset_labels], dtype=torch.float32)
    sub_sampler = torch.utils.data.WeightedRandomSampler(sub_sw, len(sub_sw), replacement=True)
    return DataLoader(subset, batch_size=batch_size,
                      sampler=sub_sampler,
                      num_workers=eff['num_workers'], pin_memory=True, drop_last=True)

print('Optimizer / curriculum utilities ready (per-model config aware).')


Optimizer / curriculum utilities ready (per-model config aware).


In [16]:
@torch.no_grad()
def evaluate_head_breakdown(model, loader, fine_crit, risk_crit, cpn_crit,
                            margin_crit, mcc_crit, riskcons_crit):
    """One forward pass over `loader`, returning the mean value of every
    loss component (fine/risk/cpn/margin/mcc/risk-consistency) alongside the
    standard classification metrics. Used for per-epoch validation logging
    of every individual head/loss curve, reusing the same forward pass
    evaluate() would otherwise do. Margin/MCC/risk-consistency all only need
    (logits, true_labels), so they are just as valid to compute on val as on
    train — they do not depend on the model being in training mode, only the
    ArcFace margin injection does (and evaluate_head_breakdown deliberately
    runs the model in eval mode, i.e. labels=None to arcface, exactly like
    every other evaluation pass in this file, for consistency)."""
    model.eval()
    total_fine=total_risk=total_cpn=total_margin=total_mcc=total_riskcons=0.; n=0
    lbls,preds,probs=[],[],[]
    for imgs,labels in loader:
        imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            fine,coarse,cpn=model(imgs)
            rt=RISK_IDX.to(DEVICE)[labels]
            fl=fine_crit(fine,labels); rl=risk_crit(coarse,rt); cl=cpn_crit(cpn,labels)
            pl=margin_crit(fine,labels); ml=mcc_crit(fine,labels); rcl=riskcons_crit(fine,labels)
        bs=imgs.size(0)
        total_fine+=fl.item()*bs; total_risk+=rl.item()*bs; total_cpn+=cl.item()*bs
        total_margin+=pl.item()*bs; total_mcc+=ml.item()*bs; total_riskcons+=rcl.item()*bs
        n+=bs
        p=torch.softmax(fine.float(),dim=1)
        lbls.extend(labels.cpu().numpy()); preds.extend(p.argmax(1).cpu().numpy()); probs.extend(p.cpu().numpy())
    met=compute_metrics(np.array(lbls),np.array(preds),np.array(probs))
    heads = {'fine':total_fine/n,'risk':total_risk/n,'cpn':total_cpn/n,
            'margin':total_margin/n,'mcc':total_mcc/n,'riskcons':total_riskcons/n}
    return heads, met


def train_model(model, train_ds, val_ds, ckpt_path, eff, task_id=None):
    """Every model is trained fully independently. `eff` is this model's
    effective config (CFG overlaid with its MODEL_OVERRIDES entry, if any).
    History (including the per-head loss breakdown) is written INSIDE
    ckpt_path on every improvement, so the checkpoint file is the single
    source of truth for this model's full training record. `task_id`, if
    given, is used to keep TaskManager status in sync (RUNNING while this
    function executes, COMPLETED/FAILED when it returns/raises) — purely for
    orchestration bookkeeping, never as a substitute for the checkpoint file
    itself as the source of truth about whether training actually finished."""
    bs = eff['batch_size']
    val_ld=DataLoader(val_ds,batch_size=bs,shuffle=False,
                      num_workers=eff['num_workers'],pin_memory=True)
    cw=train_ds.class_weights().to(DEVICE)
    fine_gamma = build_focal_gamma_tensor()  # per-class, Disease head only
    fine_crit=FocalLossLS(alpha=None,gamma=fine_gamma,smoothing=eff['label_smooth'])
    val_crit=FocalLossLS(alpha=cw,gamma=fine_gamma,smoothing=eff['label_smooth'])
    mcc_crit=SoftMCCLoss()
    rla=[RISK_OF_CLASS[CLASS_NAMES[l]] for l in train_ds.labels]
    rc=Counter(rla); nr=len(rla)
    risk_w=torch.tensor([nr/(NUM_RISK*rc.get(i,1)) for i in range(NUM_RISK)],
                         dtype=torch.float32).to(DEVICE)
    risk_crit=FocalLossLS(alpha=risk_w,gamma=eff['focal_gamma'],smoothing=eff['label_smooth'])
    cpn_crit=CPNMaskedLoss(gamma=eff['focal_gamma'],smoothing=eff['label_smooth'],
                           cpn_cw=cpn_class_weights(train_ds).to(DEVICE))
    margin_crit=ConfusablePairMarginLoss()
    riskcons_crit=RiskConsistencyLoss(gap_weights=eff['risk_consistency_gap_weights'])
    opt,sched,warmup=build_opt_sched(model,eff)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

    model.freeze_backbone()
    print(f'  Phase 1 — backbone FROZEN for {eff["phase1_epochs"]} epochs '
          f'(LoRA always active) | batch_size={bs}')
    sel=eff['selection_metric']
    assert sel == 'rec_macro', 'Contract violation: selection metric must stay val macro recall.'

    warmup_end_curriculum = int(eff['curriculum_warmup_frac'] * eff['num_epochs'])
    print(f'  Curriculum window: epochs 1-{warmup_end_curriculum} '
          f'(early-stop patience only starts counting after this window closes)')

    hist={k:[] for k in [
        'train_loss','val_loss','train_f1','val_f1','train_acc','val_acc',
        'train_rec','val_rec','train_prec','val_prec','lr','margin_weight',
        # per-head breakdown, tracked separately for individual inspection
        'train_fine_loss','train_risk_loss','train_cpn_loss',
        'train_margin_loss','train_mcc_loss','train_riskcons_loss',
        'val_fine_loss','val_risk_loss','val_cpn_loss',
        'val_margin_loss','val_mcc_loss','val_riskcons_loss',
    ]}
    best_metric=0.; patience_ctr=0
    if task_id is not None: TASKS.start(task_id, meta={'model_key':model.model_key})

    for epoch in range(1,eff['num_epochs']+1):
        t0=time.time()
        if epoch==eff['phase1_epochs']+1:
            model.unfreeze_last(n=2)
            ntr=sum(p.numel() for p in model.parameters() if p.requires_grad)
            ntot=sum(p.numel() for p in model.parameters())
            print(f'  Phase 2 — UNFROZEN last 2 stages ep {epoch} '
                  f'({ntr/1e6:.1f}M/{ntot/1e6:.1f}M trainable)')
        ph='PH1' if epoch<=eff['phase1_epochs'] else 'PH2'
        if warmup.is_warming(): warmup.step()
        else: sched.step()
        mw=margin_ramp_weight(epoch,eff['num_epochs'],eff['margin_ramp_start_frac'],
                              eff['margin_ramp_end_frac'],eff['margin_weight_start'],
                              eff['margin_weight_end'])
        train_ld=build_curriculum_loader(model,train_ds,epoch,eff['num_epochs'],bs,eff)
        model.train()
        tls=0.; tlbls,tpreds=[],[]
        sum_fine=sum_risk=sum_cpn=sum_margin=sum_mcc=sum_riskcons=0.; n_seen_loss=0

        for imgs,labels in train_ld:
            imgs,labels=imgs.to(DEVICE),labels.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            imgs_m,ya,yb,lam,do_mix=apply_mix(imgs,labels,eff['mix_prob'],eff['mixup_alpha'])
            if do_mix:
                with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                    fine,coarse,cpn=model(imgs_m,ya)
                    ra=RISK_IDX.to(DEVICE)[ya]; rb=RISK_IDX.to(DEVICE)[yb]
                    fl=mix_criterion(fine_crit,fine,ya,yb,lam)
                    rl=mix_criterion(risk_crit,coarse,ra,rb,lam)
                    cl=lam*cpn_crit(cpn,ya)+(1-lam)*cpn_crit(cpn,yb)
                    pl=torch.zeros((),device=DEVICE); ml=torch.zeros((),device=DEVICE)
                    rcl=torch.zeros((),device=DEVICE)
                    loss=fl+eff['risk_loss_weight']*rl+eff['cpn_loss_weight']*cl
            else:
                with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                    fine,coarse,cpn=model(imgs,labels)
                    rt=RISK_IDX.to(DEVICE)[labels]
                    fl=fine_crit(fine,labels)
                    rl=risk_crit(coarse,rt)
                    cl=cpn_crit(cpn,labels)
                    pl=margin_crit(fine,labels)
                    ml=mcc_crit(fine,labels)
                    rcl=riskcons_crit(fine,labels)
                    loss=(fl+eff['risk_loss_weight']*rl+eff['cpn_loss_weight']*cl
                          +mw*pl+eff['mcc_loss_weight']*ml
                          +eff['risk_consistency_weight']*rcl)
            if torch.isnan(loss) or torch.isinf(loss):
                print(f'  [WARN] NaN loss ep {epoch} — skipping batch')
                opt.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
            scaler.step(opt); scaler.update()
            bsz=imgs.size(0)
            tls+=loss.item()*bsz
            sum_fine+=fl.item()*bsz; sum_risk+=rl.item()*bsz; sum_cpn+=cl.item()*bsz
            sum_margin+=pl.item()*bsz; sum_mcc+=ml.item()*bsz; sum_riskcons+=rcl.item()*bsz
            n_seen_loss+=bsz
            tlbls.extend(ya.cpu().numpy())
            tpreds.extend(fine.detach().argmax(1).cpu().numpy())

        n_seen=len(train_ld.dataset)
        tl=tls/n_seen_loss
        tm=compute_metrics(np.array(tlbls),np.array(tpreds)) if tlbls else \
           {'f1_macro':float('nan'),'accuracy':float('nan'),
            'prec_macro':float('nan'),'rec_macro':float('nan')}
        vl,vm,_,_,_=evaluate(model,val_ld,val_crit)
        val_heads,_=evaluate_head_breakdown(model,val_ld,fine_crit,risk_crit,cpn_crit,
                                            margin_crit,mcc_crit,riskcons_crit)
        lr_now=opt.param_groups[0]['lr']
        for k,v in [('train_loss',tl),('val_loss',vl),('train_f1',tm['f1_macro']),('val_f1',vm['f1_macro']),
                    ('train_acc',tm['accuracy']),('val_acc',vm['accuracy']),
                    ('train_rec',tm['rec_macro']),('val_rec',vm['rec_macro']),
                    ('train_prec',tm['prec_macro']),('val_prec',vm['prec_macro']),
                    ('lr',lr_now),('margin_weight',mw),
                    ('train_fine_loss',sum_fine/n_seen_loss),
                    ('train_risk_loss',sum_risk/n_seen_loss),
                    ('train_cpn_loss',sum_cpn/n_seen_loss),
                    ('train_margin_loss',sum_margin/n_seen_loss),
                    ('train_mcc_loss',sum_mcc/n_seen_loss),
                    ('train_riskcons_loss',sum_riskcons/n_seen_loss),
                    ('val_fine_loss',val_heads['fine']),
                    ('val_risk_loss',val_heads['risk']),
                    ('val_cpn_loss',val_heads['cpn']),
                    ('val_margin_loss',val_heads['margin']),
                    ('val_mcc_loss',val_heads['mcc']),
                    ('val_riskcons_loss',val_heads['riskcons'])]:
            hist[k].append(v)
        cur=vm[sel]; imp='↑' if cur>best_metric else ''
        print(f'Ep {epoch:>3}/{eff["num_epochs"]} [{ph}] | '
              f'Tr L={tl:.4f} A={tm["accuracy"]:.3f} F1={tm["f1_macro"]:.3f} '
              f'P={tm["prec_macro"]:.3f} R={tm["rec_macro"]:.3f} | '
              f'Val L={vl:.4f} A={vm["accuracy"]:.3f} F1={vm["f1_macro"]:.3f} '
              f'P={vm["prec_macro"]:.3f} R={vm["rec_macro"]:.3f} | '
              f'LR={lr_now:.1e} | {time.time()-t0:.1f}s | {imp}')
        if cur>best_metric:
            best_metric,patience_ctr=cur,0
            torch.save({'epoch':epoch,
                        'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                        'best_metric':best_metric,'hist':hist,
                        'selection_metric':sel,'effective_config':eff},ckpt_path)
        else:
            if epoch > warmup_end_curriculum:
                patience_ctr+=1
                if patience_ctr>=eff['patience']:
                    print(f'  Early stop ep {epoch} (patience {eff["patience"]}, metric={sel})')
                    break
            # else: still inside the curriculum window — this epoch does not
            # count against patience, guaranteeing every model gets at least
            # `warmup_end_curriculum` fully-scored epochs before it can stop.
    print(f'  Best Val {sel}: {best_metric:.4f}')
    if task_id is not None: TASKS.complete(task_id, meta={'best_metric':best_metric,'epoch':epoch})
    return hist, ckpt_path

print('Training loop ready (per-model config, per-head loss tracking incl. RiskConsistencyLoss, curriculum-decoupled early stopping).')


Training loop ready (per-model config, per-head loss tracking incl. RiskConsistencyLoss, curriculum-decoupled early stopping).


In [17]:
# Run this once before any backbone cell. Cheap (just indexing file paths).
# Every per-model cell below also carries a defensive guard that rebuilds
# these three datasets if they aren't present in the current kernel session.
print('Building datasets...')
train_ds=LaryngoscopyDataset('train',transform=train_tf())
val_ds  =LaryngoscopyDataset('val',  transform=val_tf())
test_ds =LaryngoscopyDataset('test', transform=val_tf())
cw=train_ds.class_weights()
print('\nClass weights (higher = rarer):')
for cls,w in zip(CLASS_NAMES,cw.tolist()): print(f'  {cls:<24}: {w:.3f}')


Building datasets...
[train]  1536 images | classes: [0, 1, 2, 3, 4, 5, 6, 7]
[val  ]   416 images | classes: [0, 1, 2, 3, 4, 5, 6, 7]
[test ]   220 images | classes: [0, 1, 2, 3, 4, 5, 6, 7]

Class weights (higher = rarer):
  Cyst                    : 0.835
  Suspicion of Malignancy : 2.954
  Laryngeal cancer        : 0.800
  Leukoplakia             : 0.600
  Nodule                  : 2.233
  Papilloma               : 1.143
  Polyp                   : 0.630
  Reinke-s edema          : 1.574


In [18]:
class TaskManager:
    """Orchestration bookkeeping only — tracks which model's training run is
    WAITING/RUNNING/COMPLETED/FAILED/SKIPPED, so a long session can be
    resumed without re-running everything after an interruption. This does
    NOT replace the checkpoint file as the source of truth about whether a
    model actually finished training — train_backbone_cell still checks the
    checkpoint on disk first and keeps TaskManager's status in sync with
    that, rather than trusting TaskManager's record blindly."""
    STATUSES = ("WAITING", "RUNNING", "COMPLETED", "FAILED", "SKIPPED")

    def __init__(self, path: Path):
        self.path = path
        self.state: Dict[str, dict] = {}
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                self.state = json.load(f)

    def _save(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.state, f, indent=2, ensure_ascii=False, default=str)

    def status(self, task_id: str) -> str:
        return self.state.get(task_id, {}).get("status", "WAITING")

    def is_completed(self, task_id: str) -> bool:
        return self.status(task_id) == "COMPLETED"

    def start(self, task_id: str, meta: Optional[dict] = None):
        self.state[task_id] = {
            "status": "RUNNING", "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "meta": meta or {},
        }
        self._save()

    def complete(self, task_id: str, meta: Optional[dict] = None):
        entry = self.state.setdefault(task_id, {})
        entry["status"] = "COMPLETED"
        entry["completed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
        if meta:
            entry.setdefault("meta", {}).update(meta)
        self._save()

    def fail(self, task_id: str, reason: str = ""):
        entry = self.state.setdefault(task_id, {})
        entry["status"] = "FAILED"
        entry["reason"] = reason
        self._save()

    def skip(self, task_id: str, reason: str = ""):
        entry = self.state.setdefault(task_id, {})
        entry["status"] = "SKIPPED"
        entry["reason"] = reason
        self._save()


TASKS = TaskManager(SHARED_DIRS["logs"] / "task_status.json")
print("TaskManager ready ->", TASKS.path)


def _ensure_datasets():
    """Rebuilds the three datasets if they aren't present in this kernel
    session, so each backbone cell does not require CELL 13 to have run
    first in this specific session (only that it ran at some point —
    completed checkpoints on disk are unaffected either way)."""
    global train_ds, val_ds, test_ds
    if 'train_ds' not in globals() or 'val_ds' not in globals() or 'test_ds' not in globals():
        print('  [info] Datasets not found in this session — rebuilding...')
        train_ds=LaryngoscopyDataset('train',transform=train_tf())
        val_ds  =LaryngoscopyDataset('val',  transform=val_tf())
        test_ds =LaryngoscopyDataset('test', transform=val_tf())
    return train_ds, val_ds, test_ds


def train_backbone_cell(model_key):
    """Entry point called by each independent backbone cell. Fully
    self-contained:
      - resets seeds fresh (same result no matter what trained before it)
      - rebuilds datasets defensively if needed
      - builds this model's effective config (CFG + its override, if any)
      - skips training entirely if a valid checkpoint already exists on disk
        (the only source of truth — TaskManager status is kept in sync with
        this but never trusted on its own)
      - persists its own checkpoint + history the moment it finishes
    Safe to run in any order, any subset, any number of kernel restarts apart."""
    set_all_seeds(SEED)
    tr_ds, vl_ds, _ = _ensure_datasets()
    eff = get_model_cfg(model_key)

    d  = model_dirs(model_key)
    cp = d['checkpoints']/'best.pth'
    print('\n' + '='*65)
    print(model_key)
    print('='*65)
    overrides = MODEL_OVERRIDES.get(model_key)
    if overrides:
        print(f'  Effective config overrides: {overrides}')

    if cp.exists():
        try:
            existing = torch.load(cp, map_location='cpu', weights_only=False)
            print(f'  Checkpoint already exists — skipping training. '
                  f'(best {eff["selection_metric"]}={existing["best_metric"]:.4f}, '
                  f'stopped at epoch {existing["epoch"]})')
            print(f'  To force a retrain, delete: {cp}')
            TASKS.complete(model_key, meta={'best_metric':existing['best_metric'],
                                            'epoch':existing['epoch'],'source':'pre-existing'})
            return existing['hist']
        except Exception as ex:
            print(f'  [WARN] Existing checkpoint at {cp} failed to load ({ex}) — retraining from scratch.')

    model = LaryngoscopyModel(model_key).to(DEVICE)
    try:
        hist, _ = train_model(model, tr_ds, vl_ds, cp, eff, task_id=model_key)
    except Exception as ex:
        TASKS.fail(model_key, reason=str(ex))
        del model
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        raise
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return hist

print('train_backbone_cell() ready — one training run per model, TaskManager-tracked.')


TaskManager ready -> results_v12_9backbone_noKD_RGB\logs\task_status.json
train_backbone_cell() ready — one training run per model, TaskManager-tracked.


In [19]:
_ = train_backbone_cell('ConvNeXt-S')



ConvNeXt-S
    trying convnext_small.fb_in22k_ft_in1k ...
    loaded convnext_small.fb_in22k_ft_in1k (pretrained)
    LoRA (ConvNeXt-S): 72 layers replaced | 2.17M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.8745 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.5578 A=0.175 F1=0.166 P=0.348 R=0.245 | LR=5.0e-06 | 71.0s | ↑
Ep   2/200 [PH1] | Tr L=2.7174 A=0.002 F1=0.002 P=0.002 R=0.002 | Val L=1.3060 A=0.358 F1=0.312 P=0.353 R=0.383 | LR=1.0e-05 | 45.4s | ↑
Ep   3/200 [PH1] | Tr L=2.4478 A=0.035 F1=0.034 P=0.034 R=0.035 | Val L=1.1439 A=0.450 F1=0.448 P=0.483 R=0.502 | LR=1.5e-05 | 45.6s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.2404 A=0.105 F1=0.101 P=0.103 R=0.104 | Val L=1.0358 A=0.529 F1=0.497 P=

In [21]:
_ = train_backbone_cell('ConvNeXt-Base')



ConvNeXt-Base
  Effective config overrides: {'batch_size': 16, 'lr_backbone': 1.5e-05}
    trying convnext_base.fb_in22k_ft_in1k ...


    loaded convnext_base.fb_in22k_ft_in1k (pretrained)
    LoRA (ConvNeXt-Base): 72 layers replaced | 2.89M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=16
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.8578 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.4500 A=0.334 F1=0.319 P=0.437 R=0.395 | LR=3.8e-06 | 90.2s | ↑
Ep   2/200 [PH1] | Tr L=2.6091 A=0.012 F1=0.012 P=0.012 R=0.013 | Val L=1.2087 A=0.442 F1=0.396 P=0.444 R=0.416 | LR=7.5e-06 | 54.5s | ↑
Ep   3/200 [PH1] | Tr L=2.3226 A=0.086 F1=0.077 P=0.075 R=0.080 | Val L=1.0036 A=0.462 F1=0.463 P=0.512 R=0.508 | LR=1.1e-05 | 52.3s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=1.9674 A=0.174 F1=0.171 P=0.173 R=0.172 | Val L=0.8905 A=0.534 F1=0.534 P=0.569 R=0.545 | LR=1.5e-05 | 78.0s | ↑
Ep   5/200 [PH1] |

In [22]:
_ = train_backbone_cell('ConvFormer-S18')



ConvFormer-S18
    trying convformer_s18.sail_in22k_ft_in1k ...
    loaded convformer_s18.sail_in22k_ft_in1k (pretrained)
    LoRA (ConvFormer-S18): 72 layers replaced | 1.28M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.9857 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.6566 A=0.243 F1=0.189 P=0.382 R=0.285 | LR=5.0e-06 | 90.8s | ↑
Ep   2/200 [PH1] | Tr L=2.8506 A=0.001 F1=0.001 P=0.001 R=0.001 | Val L=1.4041 A=0.240 F1=0.203 P=0.464 R=0.299 | LR=1.0e-05 | 50.0s | ↑
Ep   3/200 [PH1] | Tr L=2.4997 A=0.023 F1=0.018 P=0.016 R=0.021 | Val L=1.2025 A=0.389 F1=0.379 P=0.472 R=0.416 | LR=1.5e-05 | 50.2s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.3164 A=0.051 F1=0.048 P=0.047 R=0.049 | Val L=1.0547 A=0.476

In [23]:
_ = train_backbone_cell('DenseNet121')



DenseNet121
  Effective config overrides: {'patience': 28, 'phase1_epochs': 16}
    trying densenet121 ...
    loaded densenet121 (pretrained)
    LoRA (DenseNet121): 61 layers replaced | 0.67M trainable params
  Phase 1 — backbone FROZEN for 16 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.9487 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.5870 A=0.276 F1=0.259 P=0.402 R=0.323 | LR=5.0e-06 | 71.8s | ↑
Ep   2/200 [PH1] | Tr L=2.8098 A=0.001 F1=0.001 P=0.001 R=0.001 | Val L=1.3660 A=0.358 F1=0.350 P=0.478 R=0.394 | LR=1.0e-05 | 45.0s | ↑
Ep   3/200 [PH1] | Tr L=2.5523 A=0.013 F1=0.012 P=0.011 R=0.013 | Val L=1.2030 A=0.447 F1=0.437 P=0.503 R=0.467 | LR=1.5e-05 | 45.1s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.4644 A=0.060 F1=0.057 P=0.055 R=0.060 | Va

In [24]:
_ = train_backbone_cell('InceptionNeXt-S')



InceptionNeXt-S
    trying inception_next_small ...
    loaded inception_next_small (pretrained)
    LoRA (InceptionNeXt-S): 72 layers replaced | 2.09M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.9692 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.6081 A=0.212 F1=0.173 P=0.272 R=0.240 | LR=5.0e-06 | 75.6s | ↑
Ep   2/200 [PH1] | Tr L=2.7907 A=0.003 F1=0.003 P=0.003 R=0.003 | Val L=1.3954 A=0.308 F1=0.263 P=0.413 R=0.321 | LR=1.0e-05 | 47.3s | ↑
Ep   3/200 [PH1] | Tr L=2.4852 A=0.028 F1=0.024 P=0.021 R=0.027 | Val L=1.2528 A=0.385 F1=0.384 P=0.453 R=0.412 | LR=1.5e-05 | 47.6s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.3339 A=0.083 F1=0.079 P=0.078 R=0.082 | Val L=1.1948 A=0.462 F1=0.420 P=0.439 R=0.43

In [25]:
_ = train_backbone_cell('HGNetV2-B4')



HGNetV2-B4
    trying hgnetv2_b4.ssld_stage2_ft_in1k ...
    loaded hgnetv2_b4.ssld_stage2_ft_in1k (pretrained)
    LoRA (HGNetV2-B4): 12 layers replaced | 0.35M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.7985 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.4338 A=0.358 F1=0.309 P=0.408 R=0.349 | LR=5.0e-06 | 69.8s | ↑
Ep   2/200 [PH1] | Tr L=2.5844 A=0.029 F1=0.023 P=0.020 R=0.029 | Val L=1.2240 A=0.428 F1=0.414 P=0.475 R=0.439 | LR=1.0e-05 | 42.1s | ↑
Ep   3/200 [PH1] | Tr L=2.3291 A=0.075 F1=0.067 P=0.063 R=0.074 | Val L=1.0914 A=0.483 F1=0.473 P=0.509 R=0.494 | LR=1.5e-05 | 40.6s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.2299 A=0.144 F1=0.138 P=0.134 R=0.146 | Val L=1.0276 A=0.524 F1=0.507 P=0.

In [26]:
_ = train_backbone_cell('EfficientNetV2-S')



EfficientNetV2-S
    trying tf_efficientnetv2_s.in21k_ft_in1k ...
    loaded tf_efficientnetv2_s.in21k_ft_in1k (pretrained)
    LoRA (EfficientNetV2-S): 68 layers replaced | 1.30M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.8976 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.5443 A=0.339 F1=0.321 P=0.362 R=0.351 | LR=5.0e-06 | 73.2s | ↑
Ep   2/200 [PH1] | Tr L=2.7511 A=0.005 F1=0.004 P=0.004 R=0.005 | Val L=1.3366 A=0.409 F1=0.387 P=0.415 R=0.411 | LR=1.0e-05 | 44.8s | ↑
Ep   3/200 [PH1] | Tr L=2.4668 A=0.043 F1=0.039 P=0.036 R=0.045 | Val L=1.1878 A=0.478 F1=0.461 P=0.481 R=0.475 | LR=1.5e-05 | 42.9s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.3382 A=0.102 F1=0.094 P=0.089 R=0.101 | Val L=1.0893 A=0

In [20]:
_ = train_backbone_cell('Swin-Tiny')



Swin-Tiny
    trying swin_tiny_patch4_window7_224.ms_in22k_ft_in1k ...
    loaded swin_tiny_patch4_window7_224.ms_in22k_ft_in1k (pretrained)
    LoRA (Swin-Tiny): 48 layers replaced | 1.13M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=24
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.8879 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.5835 A=0.284 F1=0.231 P=0.530 R=0.283 | LR=5.0e-06 | 72.6s | ↑
Ep   2/200 [PH1] | Tr L=2.7771 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.3203 A=0.394 F1=0.386 P=0.468 R=0.422 | LR=1.0e-05 | 45.0s | ↑
Ep   3/200 [PH1] | Tr L=2.4848 A=0.016 F1=0.015 P=0.014 R=0.016 | Val L=1.1589 A=0.442 F1=0.420 P=0.448 R=0.470 | LR=1.5e-05 | 44.7s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=2.2694 A=0.087 F1=0.079 P=0.079 R=0.084 | Val L=

In [27]:
_ = train_backbone_cell('DINOv2-Base')



DINOv2-Base
  Effective config overrides: {'batch_size': 12}
    trying vit_base_patch14_reg4_dinov2.lvd142m ...
    loaded vit_base_patch14_reg4_dinov2.lvd142m (pretrained)
    LoRA (DINOv2-Base): 48 layers replaced | 2.36M trainable params
  Phase 1 — backbone FROZEN for 12 epochs (LoRA always active) | batch_size=12
  Curriculum window: epochs 1-30 (early-stop patience only starts counting after this window closes)
    Curriculum 1/30: 1536/1536 (100%) kept | drop_frac target=0%
Ep   1/200 [PH1] | Tr L=2.8747 A=0.000 F1=0.000 P=0.000 R=0.000 | Val L=1.4885 A=0.236 F1=0.201 P=0.302 R=0.277 | LR=5.0e-06 | 81.5s | ↑
Ep   2/200 [PH1] | Tr L=2.6083 A=0.002 F1=0.002 P=0.002 R=0.002 | Val L=1.2984 A=0.399 F1=0.329 P=0.387 R=0.380 | LR=1.0e-05 | 57.8s | ↑
Ep   3/200 [PH1] | Tr L=2.3532 A=0.038 F1=0.035 P=0.033 R=0.038 | Val L=1.0995 A=0.454 F1=0.417 P=0.487 R=0.469 | LR=1.5e-05 | 61.1s | ↑
    Curriculum 4/30: 1483/1536 (97%) kept | drop_frac target=4%
Ep   4/200 [PH1] | Tr L=1.9484 A=0.12

In [28]:
COMPLETED = discover_completed_models()
MISSING   = [mk for mk in CFG['model_order'] if mk not in COMPLETED]

print(f'Completed models found on disk: {len(COMPLETED)}/{len(CFG["model_order"])}')
for mk in COMPLETED: print(f'  done    {mk}')
for mk in MISSING:   print(f'  pending {mk}  (run its cell above first)')

train_ds, val_ds, test_ds = _ensure_datasets()
val_ld_check = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
cw_v = val_ds.class_weights().to(DEVICE)
val_crit_check = FocalLossLS(alpha=cw_v)

MODELS = {}
for mk in COMPLETED:
    m = LaryngoscopyModel(mk, pretrained=False, verbose=False).to(DEVICE)
    ck = torch.load(checkpoint_path(mk), map_location='cpu', weights_only=False)
    m.load_state_dict(ck['model'])
    m.eval()
    ok = sanity_check_checkpoint(m, mk, val_ld_check, val_crit_check)
    MODELS[mk] = m
    if not ok:
        print(f'  -> proceeding with {mk} anyway, but treat its downstream numbers with suspicion.')

print(f'\n{len(MODELS)} model(s) loaded and ready for evaluation.')


Completed models found on disk: 9/9
  done    ConvNeXt-S
  done    ConvNeXt-Base
  done    ConvFormer-S18
  done    DenseNet121
  done    InceptionNeXt-S
  done    HGNetV2-B4
  done    EfficientNetV2-S
  done    Swin-Tiny
  done    DINOv2-Base
    LoRA (ConvNeXt-S): 72 layers replaced | 2.17M trainable params
  ✅ Sanity check OK for ConvNeXt-S: recorded=0.7751 vs fresh-eval=0.7751
    LoRA (ConvNeXt-Base): 72 layers replaced | 2.89M trainable params
  ✅ Sanity check OK for ConvNeXt-Base: recorded=0.7862 vs fresh-eval=0.7862
    LoRA (ConvFormer-S18): 72 layers replaced | 1.28M trainable params
  ✅ Sanity check OK for ConvFormer-S18: recorded=0.8017 vs fresh-eval=0.8017
    LoRA (DenseNet121): 61 layers replaced | 0.67M trainable params
  ✅ Sanity check OK for DenseNet121: recorded=0.7445 vs fresh-eval=0.7445
    LoRA (InceptionNeXt-S): 72 layers replaced | 2.09M trainable params
  ✅ Sanity check OK for InceptionNeXt-S: recorded=0.6963 vs fresh-eval=0.6963
    LoRA (HGNetV2-B4): 12 laye

In [29]:
print('Evaluating all completed models (TTA + threshold tuning + risk/disease agreement flag)...\n')
RESULTS = {}; RESULTS_THRESHOLDED = {}; THRESHOLDS = {}; RISK_FLAGS = {}

for mk, model in MODELS.items():
    met, lbl, pred, probs, risk_probs = evaluate_tta(model, test_ds)
    RESULTS[mk] = {'met': met, 'lbl': lbl, 'pred': pred, 'probs': probs, 'risk_probs': risk_probs}

    vld = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
    _, _, vl, _, vp = evaluate(model, vld, FocalLossLS(alpha=val_ds.class_weights().to(DEVICE)))
    thr = tune_thresholds(vp, vl); THRESHOLDS[mk] = thr
    tpred = apply_thresholds(probs, thr)
    tmet = compute_metrics(lbl, tpred, probs)
    RESULTS_THRESHOLDED[mk] = {'met': tmet, 'lbl': lbl, 'pred': tpred, 'probs': probs}

    # Risk/Disease agreement flag — computed against the thresholded
    # (TUNE) prediction, since that is the prediction actually used
    # downstream. Pure post-hoc check, no effect on the prediction itself.
    flags = compute_risk_disagreement(tpred, risk_probs)
    is_wrong = (lbl != tpred)
    n_flagged = int(flags.sum())
    flag_precision = float((flags & is_wrong).sum() / max(flags.sum(),1))   # of flagged, how many actually wrong
    flag_recall    = float((flags & is_wrong).sum() / max(is_wrong.sum(),1)) # of actual wrongs, how many flagged
    RISK_FLAGS[mk] = {'flags': flags, 'precision': flag_precision, 'recall': flag_recall, 'n_flagged': n_flagged}

    print(f'-- {mk} --')
    print(f'  RAW  Acc:{met["accuracy"]:.4f} F1:{met["f1_macro"]:.4f} '
          f'P:{met["prec_macro"]:.4f} R:{met["rec_macro"]:.4f} '
          f'AUC:{met.get("auc_macro",float("nan")):.4f}')
    print(f'  TUNE Acc:{tmet["accuracy"]:.4f} F1:{tmet["f1_macro"]:.4f} '
          f'P:{tmet["prec_macro"]:.4f} R:{tmet["rec_macro"]:.4f} '
          f'AUC:{tmet.get("auc_macro",float("nan")):.4f}')
    print(f'  Risk/Disease disagreement flag: {n_flagged}/{len(lbl)} flagged | '
          f'precision={flag_precision:.3f} (of flagged, fraction actually wrong) | '
          f'recall={flag_recall:.3f} (of actual errors, fraction caught by the flag)\n')

    rd = model_dirs(mk)['test_report']
    with open(rd/'metrics.json', 'w') as f:
        json.dump({'raw': {k:v for k,v in met.items() if k not in ('f1_per_class','sensitivity','specificity')},
                   'tune': {k:v for k,v in tmet.items() if k not in ('f1_per_class','sensitivity','specificity')},
                   'thresholds': {str(k):v for k,v in thr.items()},
                   'risk_disease_flag': {'n_flagged':n_flagged,'precision':flag_precision,'recall':flag_recall}},
                  f, indent=2, default=float)
    np.savez(rd/'arrays.npz', raw_lbl=lbl, raw_pred=pred, raw_probs=probs, tune_pred=tpred,
             risk_probs=risk_probs, risk_disagreement_flag=flags)

WINNER_KEY  = max(MODELS, key=lambda k: RESULTS_THRESHOLDED[k]['met']['rec_macro'])
WINNER_DIRS = model_dirs(WINNER_KEY)
print(f'Done. Results for {len(MODELS)} model(s) saved under '
      f'{CFG["out_dir"]}/models/<name>/test_report/.')
print(f'Best model by TUNE macro recall: {WINNER_KEY}')


Evaluating all completed models (TTA + threshold tuning + risk/disease agreement flag)...



-- ConvNeXt-S --
  RAW  Acc:0.8091 F1:0.8022 P:0.7962 R:0.8257 AUC:0.9698
  TUNE Acc:0.8136 F1:0.7978 P:0.8044 R:0.8111 AUC:0.9698
  Risk/Disease disagreement flag: 21/220 flagged | precision=0.476 (of flagged, fraction actually wrong) | recall=0.244 (of actual errors, fraction caught by the flag)



-- ConvNeXt-Base --
  RAW  Acc:0.8318 F1:0.8282 P:0.8365 R:0.8267 AUC:0.9569
  TUNE Acc:0.8273 F1:0.8240 P:0.8320 R:0.8240 AUC:0.9569
  Risk/Disease disagreement flag: 5/220 flagged | precision=0.600 (of flagged, fraction actually wrong) | recall=0.079 (of actual errors, fraction caught by the flag)



-- ConvFormer-S18 --
  RAW  Acc:0.7955 F1:0.7842 P:0.7900 R:0.7884 AUC:0.9726
  TUNE Acc:0.7818 F1:0.7698 P:0.7787 R:0.7728 AUC:0.9726
  Risk/Disease disagreement flag: 6/220 flagged | precision=0.500 (of flagged, fraction actually wrong) | recall=0.062 (of actual errors, fraction caught by the flag)



-- DenseNet121 --
  RAW  Acc:0.7455 F1:0.7358 P:0.7287 R:0.7599 AUC:0.9601
  TUNE Acc:0.7545 F1:0.7395 P:0.7373 R:0.7603 AUC:0.9601
  Risk/Disease disagreement flag: 7/220 flagged | precision=0.429 (of flagged, fraction actually wrong) | recall=0.056 (of actual errors, fraction caught by the flag)



-- InceptionNeXt-S --
  RAW  Acc:0.7636 F1:0.7375 P:0.7480 R:0.7604 AUC:0.9610
  TUNE Acc:0.7455 F1:0.7303 P:0.7423 R:0.7498 AUC:0.9610
  Risk/Disease disagreement flag: 30/220 flagged | precision=0.567 (of flagged, fraction actually wrong) | recall=0.304 (of actual errors, fraction caught by the flag)



-- HGNetV2-B4 --
  RAW  Acc:0.8045 F1:0.7855 P:0.7937 R:0.8121 AUC:0.9730
  TUNE Acc:0.8045 F1:0.7922 P:0.8010 R:0.8125 AUC:0.9730
  Risk/Disease disagreement flag: 12/220 flagged | precision=0.583 (of flagged, fraction actually wrong) | recall=0.163 (of actual errors, fraction caught by the flag)



-- EfficientNetV2-S --
  RAW  Acc:0.7909 F1:0.7851 P:0.7715 R:0.8087 AUC:0.9663
  TUNE Acc:0.7864 F1:0.7840 P:0.7719 R:0.8045 AUC:0.9663
  Risk/Disease disagreement flag: 8/220 flagged | precision=0.625 (of flagged, fraction actually wrong) | recall=0.106 (of actual errors, fraction caught by the flag)



-- Swin-Tiny --
  RAW  Acc:0.8364 F1:0.8355 P:0.8291 R:0.8600 AUC:0.9802
  TUNE Acc:0.8318 F1:0.8331 P:0.8366 R:0.8450 AUC:0.9802
  Risk/Disease disagreement flag: 13/220 flagged | precision=0.308 (of flagged, fraction actually wrong) | recall=0.108 (of actual errors, fraction caught by the flag)



-- DINOv2-Base --
  RAW  Acc:0.8182 F1:0.8087 P:0.8171 R:0.8099 AUC:0.9791
  TUNE Acc:0.8227 F1:0.8133 P:0.8212 R:0.8133 AUC:0.9791
  Risk/Disease disagreement flag: 9/220 flagged | precision=0.333 (of flagged, fraction actually wrong) | recall=0.077 (of actual errors, fraction caught by the flag)

Done. Results for 9 model(s) saved under results_v12_9backbone_noKD_RGB/models/<name>/test_report/.
Best model by TUNE macro recall: Swin-Tiny


In [30]:
if len(MODELS) > 0:
    names = list(MODELS.keys())
    n_flagged = [RISK_FLAGS[n]['n_flagged'] for n in names]
    precisions = [RISK_FLAGS[n]['precision'] for n in names]
    recalls    = [RISK_FLAGS[n]['recall'] for n in names]
    n_total = len(RESULTS_THRESHOLDED[names[0]]['lbl'])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Risk/Disease Agreement Flag — quality as an error detector '
                f'(no effect on predictions, post-hoc consistency check only, n={n_total} test images)',
                fontsize=12, fontweight='bold')

    xi = np.arange(len(names))
    axes[0].bar(xi, n_flagged, color='#f59e0b', alpha=0.85)
    for i,v in enumerate(n_flagged):
        axes[0].text(i, v+0.3, str(v), ha='center', fontsize=9, fontweight='bold')
    axes[0].set_xticks(xi); axes[0].set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    axes[0].set_ylabel('Number of test images flagged')
    axes[0].set_title('How many samples get flagged per model', fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].bar(xi-0.15, precisions, 0.3, label='Precision (flagged -> actually wrong)', color='#dc2626', alpha=0.85)
    axes[1].bar(xi+0.15, recalls,    0.3, label='Recall (actual errors -> caught by flag)', color='#2563eb', alpha=0.85)
    for i,(p,r) in enumerate(zip(precisions,recalls)):
        axes[1].text(i-0.15, p+0.02, f'{p:.2f}', ha='center', fontsize=8)
        axes[1].text(i+0.15, r+0.02, f'{r:.2f}', ha='center', fontsize=8)
    axes[1].set_xticks(xi); axes[1].set_xticklabels(names, rotation=30, ha='right', fontsize=9)
    axes[1].set_ylim(0,1.1); axes[1].legend(fontsize=9); axes[1].grid(axis='y', alpha=0.3)
    axes[1].set_title('Flag quality as an error detector', fontweight='bold')

    plt.tight_layout()
    fname = SHARED_DIRS['comparison']/'risk_disease_flag_quality.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight'); plt.close()
    print(f'Saved: {fname}')
    print('\nReading this: precision tells you how often a flagged case is worth a second look '
          '(high precision = the flag is not just noise). Recall tells you what fraction of the '
          'model\'s actual mistakes this flag would have caught if used as a review trigger.')


Saved: results_v12_9backbone_noKD_RGB\comparison\risk_disease_flag_quality.png

Reading this: precision tells you how often a flagged case is worth a second look (high precision = the flag is not just noise). Recall tells you what fraction of the model's actual mistakes this flag would have caught if used as a review trigger.


In [31]:
def plot_dual_confusion_matrices(mk, out_dir):
    r=RESULTS_THRESHOLDED[mk]; lbl,pred=r['lbl'],r['pred']
    cm=confusion_matrix(lbl,pred,labels=list(range(NUM_CLASSES)))
    cm_n=cm.astype(float)/(cm.sum(axis=1,keepdims=True)+1e-9)
    fig,axes=plt.subplots(1,2,figsize=(22,9))
    fig.suptitle(f'Confusion Matrices — {mk} (TTA + Thresholds)',fontsize=15,fontweight='bold')
    short=[c[:14] for c in CLASS_NAMES]
    specs=[(axes[0],cm,'d','Blues','Raw Counts\n(how many images per cell)',cm.max()),
           (axes[1],cm_n,'.2f','Greens',
            'Normalised (Recall per Row)\nDiagonal = class recall; off-diagonal = confusion rate',1.0)]
    for ax,data,fmt,cmap,title,vmax in specs:
        sns.heatmap(data,annot=True,fmt=fmt,cmap=cmap,xticklabels=short,yticklabels=short,
                    ax=ax,linewidths=0.4,vmin=0,vmax=vmax,annot_kws={'size':9})
        for i in range(NUM_CLASSES):
            ax.add_patch(plt.Rectangle((i,i),1,1,fill=False,edgecolor='#f0a500',lw=2.5))
        ax.set_title(title,fontweight='bold',pad=10)
        ax.set_xlabel('Predicted Class'); ax.set_ylabel('True Class')
        ax.tick_params(axis='x',rotation=40,labelsize=9)
        ax.tick_params(axis='y',rotation=0, labelsize=9)
    plt.tight_layout()
    fname=out_dir/f'confusion_matrices_{model_key_safe(mk)}.png'
    plt.savefig(fname,dpi=150,bbox_inches='tight'); plt.close()
    print(f'  Saved: {fname}')


def full_evaluation_report(mk,out_dir):
    r=RESULTS_THRESHOLDED[mk]; bm,bl,bp,bpr=r['met'],r['lbl'],r['pred'],r['probs']
    fig=plt.figure(figsize=(22,14))
    fig.suptitle(f'Evaluation — {mk}\nF1:{bm["f1_macro"]:.4f} Acc:{bm["accuracy"]:.4f} '
                 f'Rec:{bm["rec_macro"]:.4f} AUC:{bm.get("auc_macro",float("nan")):.4f}',
                 fontsize=14,fontweight='bold')
    gs=gridspec.GridSpec(2,2,hspace=0.45,wspace=0.35)
    ax_roc=fig.add_subplot(gs[0,:])
    cols=plt.cm.Set2(np.linspace(0,1,NUM_CLASSES))
    for i,(cls,col) in enumerate(zip(CLASS_NAMES,cols)):
        try:
            fpr,tpr,_=roc_curve((bl==i).astype(int),bpr[:,i])
            ai=roc_auc_score((bl==i).astype(int),bpr[:,i])
            ax_roc.plot(fpr,tpr,color=col,lw=2,label=f'{cls[:14]} (AUC={ai:.2f})')
        except Exception: pass
    ax_roc.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
    ax_roc.set_title('ROC Curves — One-vs-Rest',fontweight='bold')
    ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
    ax_roc.legend(fontsize=8,ncol=4,loc='lower right'); ax_roc.grid(alpha=0.3)
    ax_prf=fig.add_subplot(gs[1,0])
    prec=precision_score(bl,bp,average=None,zero_division=0)
    rec =recall_score(   bl,bp,average=None,zero_division=0)
    f1s =np.array(bm['f1_per_class'])
    x=np.arange(NUM_CLASSES); w=0.25
    for vals,off,col,lbl2 in [(prec,-w,'steelblue','Precision'),(rec,0,'darkorange','Recall'),(f1s,w,'seagreen','F1')]:
        bars=ax_prf.bar(x+off,vals,w,color=col,alpha=0.85,label=lbl2)
        for b in bars:
            h=b.get_height()
            if h>0.02: ax_prf.text(b.get_x()+b.get_width()/2,h+0.01,f'{h:.2f}',
                                    ha='center',fontsize=6,fontweight='bold')
    ax_prf.set_xticks(x); ax_prf.set_xticklabels(CLASS_NAMES,rotation=38,ha='right',fontsize=8)
    ax_prf.set_ylim(0,1.15); ax_prf.legend(fontsize=8); ax_prf.grid(axis='y',alpha=0.3)
    ax_prf.set_title('Per-Class Precision / Recall / F1',fontweight='bold')
    ax_ss=fig.add_subplot(gs[1,1]); yp=np.arange(NUM_CLASSES)
    ax_ss.barh(yp-0.2,bm['sensitivity'],0.35,label='Sensitivity',color='tomato',   alpha=0.85)
    ax_ss.barh(yp+0.2,bm['specificity'],0.35,label='Specificity',color='royalblue',alpha=0.85)
    ax_ss.set_yticks(yp); ax_ss.set_yticklabels([c[:14] for c in CLASS_NAMES],fontsize=8)
    ax_ss.set_xlim(0,1.12); ax_ss.legend(fontsize=8); ax_ss.grid(axis='x',alpha=0.3)
    ax_ss.set_title('Sensitivity / Specificity',fontweight='bold')
    fname=out_dir/f'evaluation_report_{model_key_safe(mk)}.png'
    plt.savefig(fname,dpi=150,bbox_inches='tight'); plt.close(); print(f'  Saved: {fname}')


print('Dual confusion matrices for every completed model...')
for mk in MODELS: plot_dual_confusion_matrices(mk,model_dirs(mk)['test_report'])

print('\nFull evaluation reports for every completed model...')
for mk in MODELS: full_evaluation_report(mk,model_dirs(mk)['test_report'])

print('\nClassification reports (thresholded TTA):')
for mk in MODELS:
    r=RESULTS_THRESHOLDED[mk]
    print('\n'+'='*70)
    print(f'CLASSIFICATION REPORT — {mk}'); print('='*70)
    print(classification_report(r['lbl'],r['pred'],target_names=CLASS_NAMES,digits=4,zero_division=0))


Dual confusion matrices for every completed model...
  Saved: results_v12_9backbone_noKD_RGB\models\ConvNeXt_S\test_report\confusion_matrices_ConvNeXt_S.png
  Saved: results_v12_9backbone_noKD_RGB\models\ConvNeXt_Base\test_report\confusion_matrices_ConvNeXt_Base.png
  Saved: results_v12_9backbone_noKD_RGB\models\ConvFormer_S18\test_report\confusion_matrices_ConvFormer_S18.png
  Saved: results_v12_9backbone_noKD_RGB\models\DenseNet121\test_report\confusion_matrices_DenseNet121.png
  Saved: results_v12_9backbone_noKD_RGB\models\InceptionNeXt_S\test_report\confusion_matrices_InceptionNeXt_S.png
  Saved: results_v12_9backbone_noKD_RGB\models\HGNetV2_B4\test_report\confusion_matrices_HGNetV2_B4.png
  Saved: results_v12_9backbone_noKD_RGB\models\EfficientNetV2_S\test_report\confusion_matrices_EfficientNetV2_S.png
  Saved: results_v12_9backbone_noKD_RGB\models\Swin_Tiny\test_report\confusion_matrices_Swin_Tiny.png
  Saved: results_v12_9backbone_noKD_RGB\models\DINOv2_Base\test_report\confusio

In [32]:
names = list(MODELS.keys())
if len(names) == 0:
    print('No completed models yet — nothing to compare. Train at least one backbone first.')
else:
    f1_v  = [RESULTS_THRESHOLDED[n]["met"]["f1_macro"] for n in names]
    acc_v = [RESULTS_THRESHOLDED[n]["met"]["accuracy"]  for n in names]
    rec_v = [RESULTS_THRESHOLDED[n]["met"]["rec_macro"] for n in names]
    auc_v = [RESULTS_THRESHOLDED[n]["met"].get("auc_macro", 0) for n in names]

    fig, ax = plt.subplots(figsize=(max(14, 1.8*len(names)), 6))
    xi = np.arange(len(names))
    ax.bar(xi - 0.3, f1_v,  0.2, label="Macro F1",     color="steelblue",   alpha=0.85)
    ax.bar(xi - 0.1, acc_v, 0.2, label="Accuracy",     color="seagreen",    alpha=0.85)
    ax.bar(xi + 0.1, rec_v, 0.2, label="Macro Recall", color="darkorange",  alpha=0.85)
    ax.bar(xi + 0.3, auc_v, 0.2, label="AUC",          color="mediumpurple",alpha=0.85)
    for i,(f,a,r,au) in enumerate(zip(f1_v,acc_v,rec_v,auc_v)):
        for off,v in zip([-0.3,-0.1,0.1,0.3],[f,a,r,au]):
            ax.text(i+off,v+0.01,f'{v:.3f}',ha='center',fontsize=7,fontweight='bold',rotation=90)
    xlbls = [f'{n}\n(BEST)' if n==WINNER_KEY else n for n in names]
    ax.set_xticks(xi); ax.set_xticklabels(xlbls, fontsize=9)
    ax.set_ylim(0, 1.22); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"Model Comparison — thresholded predictions, TTA "
                 f"({len(names)}/{len(CFG['model_order'])} backbones completed)",
                 fontweight="bold")
    plt.tight_layout()
    fname = SHARED_DIRS['comparison']/'model_comparison.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {fname}')


Saved: results_v12_9backbone_noKD_RGB\comparison\model_comparison.png


In [33]:
METRIC_PAIRS=[
    ('train_loss','val_loss','Loss'),
    ('train_acc', 'val_acc', 'Accuracy'),
    ('train_f1',  'val_f1',  'Macro_F1'),
    ('train_prec','val_prec','Macro_Precision'),
    ('train_rec', 'val_rec', 'Macro_Recall'),
]

def plot_training_curves(mk, hist, curves_dir, phase1_epochs):
    ph2=phase1_epochs+1
    ep=list(range(1,len(hist['train_loss'])+1))
    for tr_k,vl_k,label in METRIC_PAIRS:
        if not hist.get(tr_k): continue
        fig,ax=plt.subplots(figsize=(10,5))
        ax.plot(ep,hist[tr_k],'-o',color='#7b9ecf',ms=3,lw=1.8,label='Train')
        ax.plot(ep,hist[vl_k],'-o',color='#e05c5c',ms=3,lw=1.8,label='Val')
        if ph2<=max(ep):
            ax.axvline(x=ph2-0.5,color='#d4c000',lw=1.5,linestyle='--',label='Phase 2 start')
        if 'Loss' not in label: ax.set_ylim(0,1)
        ax.set_xlim(min(ep)-0.1,max(ep)+0.1)
        ax.set_xlabel('Epoch',fontsize=11); ax.set_ylabel(label.replace('_',' '),fontsize=11)
        ax.set_title(f'{mk} — {label.replace("_"," ")}',fontweight='bold',fontsize=13)
        ax.legend(fontsize=10); ax.grid(alpha=0.25)
        plt.tight_layout()
        fname=curves_dir/f'curves_{model_key_safe(mk)}_{label}.png'
        plt.savefig(fname,dpi=150,bbox_inches='tight'); plt.close()
    print(f'  {mk}: {len(METRIC_PAIRS)} individual curve images -> {curves_dir}')


print('Learning curves (one image per metric) for every completed model...')
for mk in discover_completed_models():
    eff = get_model_cfg(mk)
    plot_training_curves(mk, load_history(mk), model_dirs(mk)['curves'], eff['phase1_epochs'])


Learning curves (one image per metric) for every completed model...
  ConvNeXt-S: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\ConvNeXt_S\curves
  ConvNeXt-Base: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\ConvNeXt_Base\curves
  ConvFormer-S18: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\ConvFormer_S18\curves
  DenseNet121: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\DenseNet121\curves
  InceptionNeXt-S: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\InceptionNeXt_S\curves
  HGNetV2-B4: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\HGNetV2_B4\curves
  EfficientNetV2-S: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\EfficientNetV2_S\curves
  Swin-Tiny: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\Swin_Tiny\curves
  DINOv2-Base: 5 individual curve images -> results_v12_9backbone_noKD_RGB\models\DINOv2_Base\curves


In [34]:
HEAD_LOSS_SPECS = [
    # (train_key, val_key, display name) — every component now has both,
    # each gets its own plot, no component's curve is drawn on top of another's.
    ('train_fine_loss',     'val_fine_loss',     'Disease head loss (fine)'),
    ('train_risk_loss',     'val_risk_loss',     'Risk head loss (coarse)'),
    ('train_cpn_loss',      'val_cpn_loss',      'CPN head loss (Cyst/Nodule/Polyp)'),
    ('train_margin_loss',   'val_margin_loss',   'Confusable-pair margin loss'),
    ('train_mcc_loss',      'val_mcc_loss',      'Soft-MCC loss'),
    ('train_riskcons_loss', 'val_riskcons_loss', 'Risk-consistency loss'),
]

def plot_individual_head_losses(mk, hist, curves_dir):
    """Each loss component plotted on its own — this is the per-head view
    requested before looking at how they combine."""
    ep=list(range(1,len(hist['train_loss'])+1))
    for tr_k, vl_k, name in HEAD_LOSS_SPECS:
        if not hist.get(tr_k): continue
        fig,ax=plt.subplots(figsize=(9,4.6))
        ax.plot(ep,hist[tr_k],'-o',ms=3,lw=1.6,color='steelblue',label='Train')
        if vl_k and hist.get(vl_k):
            ax.plot(ep,hist[vl_k],'-o',ms=3,lw=1.6,color='#e05c5c',label='Val')
        ax.set_title(f'{mk} — {name}',fontweight='bold',fontsize=12)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
        plt.tight_layout()
        safe_name=re.sub(r'[^A-Za-z0-9]+','_',name).strip('_')
        fname=curves_dir/f'headloss_{model_key_safe(mk)}_{safe_name}.png'
        plt.savefig(fname,dpi=150,bbox_inches='tight'); plt.close()


def plot_combined_head_losses(mk, hist, curves_dir, eff):
    """All head-loss components together with their configured weights
    applied, alongside the total training loss — shows how the individual
    losses plotted above actually sum into the number the optimiser sees.
    Weighted contributions: fine (weight 1.0, the base loss), risk
    (risk_loss_weight), cpn (cpn_loss_weight), margin (the ramped
    margin_weight schedule, which changes value epoch to epoch), mcc
    (mcc_loss_weight), and risk-consistency (risk_consistency_weight)."""
    ep=list(range(1,len(hist['train_loss'])+1))
    fine   = np.array(hist['train_fine_loss'])
    risk_w = np.array(hist['train_risk_loss'])   * eff['risk_loss_weight']
    cpn_w  = np.array(hist['train_cpn_loss'])    * eff['cpn_loss_weight']
    margin_w = np.array(hist['train_margin_loss']) * np.array(hist['margin_weight'])
    mcc_w  = np.array(hist['train_mcc_loss'])    * eff['mcc_loss_weight']
    riskcons_w = np.array(hist['train_riskcons_loss']) * eff['risk_consistency_weight']

    fig,ax=plt.subplots(figsize=(10,5.5))
    ax.stackplot(ep, fine, risk_w, cpn_w, margin_w, mcc_w, riskcons_w,
                 labels=['Disease (fine) x1.0',
                         f'Risk x{eff["risk_loss_weight"]}',
                         f'CPN x{eff["cpn_loss_weight"]}',
                         'Margin x(ramped weight)',
                         f'Soft-MCC x{eff["mcc_loss_weight"]}',
                         f'Risk-consistency x{eff["risk_consistency_weight"]}'],
                 colors=['#4c72b0','#dd8452','#55a868','#c44e52','#8172b2','#937860'],
                 alpha=0.85)
    ax.plot(ep,hist['train_loss'],'k--',lw=1.8,label='Total train loss (actual)')
    ax.set_title(f'{mk} — Combined loss composition',fontweight='bold',fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted loss contribution')
    ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.25)
    plt.tight_layout()
    fname=curves_dir/f'headloss_{model_key_safe(mk)}_combined.png'
    plt.savefig(fname,dpi=150,bbox_inches='tight'); plt.close()


def plot_margin_schedule(mk, hist, curves_dir):
    fig,ax=plt.subplots(figsize=(8,4.2))
    ep=range(1,len(hist['train_loss'])+1)
    ax.plot(ep,hist['margin_weight'],'-',lw=1.8,color='teal')
    ax.set_title(f'{mk} — Margin weight schedule',fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Margin weight'); ax.grid(alpha=0.3)
    plt.tight_layout()
    fname=curves_dir/f'margin_schedule_{model_key_safe(mk)}.png'
    plt.savefig(fname,dpi=150,bbox_inches='tight'); plt.close()


print('Per-head loss curves (individual, then combined) for every completed model...')
for mk in discover_completed_models():
    eff = get_model_cfg(mk)
    hist = load_history(mk)
    d = model_dirs(mk)['curves']
    plot_individual_head_losses(mk, hist, d)
    plot_combined_head_losses(mk, hist, d, eff)
    plot_margin_schedule(mk, hist, d)
    print(f'  {mk}: done.')


Per-head loss curves (individual, then combined) for every completed model...
  ConvNeXt-S: done.
  ConvNeXt-Base: done.
  ConvFormer-S18: done.
  DenseNet121: done.
  InceptionNeXt-S: done.
  HGNetV2-B4: done.
  EfficientNetV2-S: done.
  Swin-Tiny: done.
  DINOv2-Base: done.


In [35]:
ERROR_PAIRS_TOP_K=6; ERROR_EXAMPLES_PER_PAIR=4

def top_confusion_pairs(lbl,pred,probs,k):
    counts=Counter(); confs=defaultdict(list)
    for i in range(len(lbl)):
        if lbl[i]!=pred[i]:
            pair=(CLASS_NAMES[lbl[i]],CLASS_NAMES[pred[i]])
            counts[pair]+=1; confs[pair].append(float(probs[i,pred[i]]))
    return [(p,c,float(np.mean(confs[p]))) for p,c in counts.most_common(k)]

def error_overview(mk,out_dir):
    r=RESULTS_THRESHOLDED[mk]; lbl,pred,probs=r['lbl'],r['pred'],r['probs']
    is_w=lbl!=pred
    print(f'  {mk}: errors={is_w.sum()}/{len(lbl)} ({is_w.mean()*100:.1f}%)')
    wc=probs[np.arange(len(pred)),pred][is_w]
    tc=probs[np.arange(len(pred)),lbl][is_w]
    pairs=top_confusion_pairs(lbl,pred,probs,ERROR_PAIRS_TOP_K)
    fig,axes=plt.subplots(1,2,figsize=(16,6))
    fig.suptitle(f'Error Analysis — {mk}',fontsize=14,fontweight='bold')
    bins=np.linspace(0,1,26)
    axes[0].hist(tc,bins=bins,alpha=0.7,color='#16a34a',label='True class confidence')
    axes[0].hist(wc,bins=bins,alpha=0.7,color='#ef4444',label='Wrong class confidence')
    axes[0].set_xlabel('Softmax probability'); axes[0].set_ylabel('Count')
    axes[0].set_title('Error confidence distribution',fontweight='bold'); axes[0].legend(fontsize=9)
    if pairs:
        pl=[f'{t}\n-> {p}' for (t,p),_,_ in pairs][::-1]
        cnts=[c for _,c,_ in pairs][::-1]
        bars=axes[1].barh(pl,cnts,color='#f59e0b'); axes[1].bar_label(bars,fontsize=8,padding=2)
    axes[1].set_xlabel('Count'); axes[1].set_title('Most frequent confusion pairs',fontweight='bold')
    axes[1].tick_params(axis='y',labelsize=8)
    fig.tight_layout()
    fig.savefig(out_dir/f'error_overview_{model_key_safe(mk)}.png',bbox_inches='tight'); plt.close()

def confusion_pair_details(mk,out_dir):
    r=RESULTS_THRESHOLDED[mk]; lbl,pred,probs=r['lbl'],r['pred'],r['probs']
    detail_dir=out_dir/'confusion_pairs'; detail_dir.mkdir(exist_ok=True)
    p2i=defaultdict(list)
    for i in range(len(lbl)):
        if lbl[i]!=pred[i]: p2i[(int(lbl[i]),int(pred[i]))].append(i)
    ranked=sorted(p2i.items(),key=lambda kv:len(kv[1]),reverse=True)[:ERROR_PAIRS_TOP_K]
    for rank,((ti,pi),idxs) in enumerate(ranked,1):
        tn,pn=CLASS_NAMES[ti],CLASS_NAMES[pi]
        avg_conf=float(np.mean(probs[idxs,pi]))
        tr,pr=RISK_OF_CLASS[tn],RISK_OF_CLASS[pn]
        if pr<tr:
            risk_note=(f'RISK-DOWNGRADING: true={RISK_NAMES[tr]} -> predicted={RISK_NAMES[pr]}.\n'
                       f'"{tn}" is {RISK_NAMES[tr]}; predicted "{pn}" is {RISK_NAMES[pr]}.')
        else:
            risk_note=(f'Both classes are in the same or not-lower risk group '
                       f'({RISK_NAMES[tr]} -> {RISK_NAMES[pr]}).\n'
                       f'Both "{tn}" and "{pn}" fall under: '
                       f'{RISK_NAMES[tr]} / {RISK_NAMES[pr]} respectively.')
        n_ex=min(ERROR_EXAMPLES_PER_PAIR,len(idxs)); ex_idxs=idxs[:n_ex]
        fig=plt.figure(figsize=(4*n_ex+4.5,6.5))
        grid=gridspec.GridSpec(2,n_ex+1,width_ratios=[1]*n_ex+[1.7],
                               height_ratios=[3,1],hspace=0.12,wspace=0.25)
        fig.suptitle(f'{mk} — Pair #{rank}: True {tn} -> Predicted {pn}\n'
                     f'({len(idxs)} errors, avg confidence {avg_conf:.1%})',
                     fontsize=12,fontweight='bold')
        for col,idx in enumerate(ex_idxs):
            path,_=test_ds.samples[idx]
            img=Image.open(path).convert('RGB').resize((CFG['img_size'],CFG['img_size']))
            ax_img=fig.add_subplot(grid[0,col])
            ax_img.imshow(img); ax_img.axis('off')
            ax_img.set_title(f'pred conf={probs[idx,pi]:.0%}\ntrue-class p={probs[idx,ti]:.0%}',fontsize=8)
        ax_txt=fig.add_subplot(grid[1,:n_ex]); ax_txt.axis('off')
        ax_txt.text(0.,0.92,f'Errors: {len(idxs)}   Avg. confidence in wrong class: {avg_conf:.1%}\n{risk_note}',
                    va='top',fontsize=9,bbox=dict(boxstyle='round',facecolor='#fef3c7',edgecolor='#f59e0b'))
        ax_bar=fig.add_subplot(grid[:,n_ex])
        avg_p=probs[idxs].mean(axis=0)
        colors=['#2563eb' if i==ti else ('#dc2626' if i==pi else '#cbd5e1') for i in range(NUM_CLASSES)]
        bars2=ax_bar.barh(CLASS_NAMES,avg_p,color=colors)
        ax_bar.bar_label(bars2,fmt='%.2f',fontsize=8,padding=2)
        ax_bar.set_xlim(0,1.0)
        ax_bar.set_title('Avg. class probability across these errors\n(blue=true, red=predicted)',fontsize=9)
        ax_bar.tick_params(labelsize=8)
        fig.tight_layout()
        fname=detail_dir/f'pair_{rank:02d}_{tn.replace(" ","_")}_to_{pn.replace(" ","_")}.png'
        fig.savefig(fname,bbox_inches='tight',dpi=130); plt.close()
    print(f'  {mk}: {len(ranked)} confusion-pair images -> {detail_dir}')

print('Error analysis for every completed model...')
for mk in MODELS:
    d=model_dirs(mk)['error_analysis']
    error_overview(mk,d); confusion_pair_details(mk,d)


Error analysis for every completed model...
  ConvNeXt-S: errors=41/220 (18.6%)
  ConvNeXt-S: 6 confusion-pair images -> results_v12_9backbone_noKD_RGB\models\ConvNeXt_S\error_analysis\confusion_pairs
  ConvNeXt-Base: errors=38/220 (17.3%)
  ConvNeXt-Base: 6 confusion-pair images -> results_v12_9backbone_noKD_RGB\models\ConvNeXt_Base\error_analysis\confusion_pairs
  ConvFormer-S18: errors=48/220 (21.8%)
  ConvFormer-S18: 6 confusion-pair images -> results_v12_9backbone_noKD_RGB\models\ConvFormer_S18\error_analysis\confusion_pairs
  DenseNet121: errors=54/220 (24.5%)
  DenseNet121: 6 confusion-pair images -> results_v12_9backbone_noKD_RGB\models\DenseNet121\error_analysis\confusion_pairs
  InceptionNeXt-S: errors=56/220 (25.5%)
  InceptionNeXt-S: 6 confusion-pair images -> results_v12_9backbone_noKD_RGB\models\InceptionNeXt_S\error_analysis\confusion_pairs
  HGNetV2-B4: errors=43/220 (19.5%)
  HGNetV2-B4: 6 confusion-pair images -> results_v12_9backbone_noKD_RGB\models\HGNetV2_B4\error_

In [36]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from captum.attr import IntegratedGradients, Occlusion

class FineHeadWrapper(nn.Module):
    """Exposes only the Disease (fine) head output, so attribution methods
    that expect a single-tensor-output classifier can be pointed at it
    directly, regardless of the underlying model's three-head structure."""
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,x): fine,_,_=self.m(x); return fine

def _target_layer(model):
    inner=getattr(model.backbone,'model',model.backbone)
    for attr in ('stages','blocks','layers'):
        if hasattr(inner,attr): return [list(getattr(inner,attr))[-1]]
    if hasattr(inner,'features'): return [list(inner.features.children())[-1]]
    return [inner]

def _normalize_map(a):
    a=a-a.min()
    m=a.max()
    return a/m if m>1e-8 else a

def pick_one_case_per_class(mk, mode):
    """One representative test-set case per disease class, for a given
    correctness mode ('correct' or 'wrong'). Returns {class_idx: sample_idx}."""
    r=RESULTS_THRESHOLDED[mk]; lbl,pred=r['lbl'],r['pred']
    mask=(lbl==pred) if mode=='correct' else (lbl!=pred)
    chosen={}
    for ci in range(NUM_CLASSES):
        cands=np.where(mask&(lbl==ci))[0]
        if len(cands)>0: chosen[ci]=int(cands[0])
    return chosen

def save_xai_panel(mk, technique_name, mode, class_name, image_tensor,
                   true_idx, target_idx, probs_row, heatmap_2d, overlay_rgb,
                   out_path, heatmap_cmap):
    """One saved image per case: Original | Heatmap | Overlay | Class
    probabilities — the same four-panel layout for every technique and
    every model."""
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(f'{technique_name} on {mode.upper()} Predictions — {mk}',
                fontsize=14, fontweight='bold')

    orig = denorm(image_tensor).permute(1,2,0).cpu().numpy()
    axes[0].imshow(orig); axes[0].axis('off')
    title0 = f'True: {class_name}'
    if mode=='wrong': title0 += f'\nPred: {CLASS_NAMES[target_idx]}'
    axes[0].set_title(title0, fontsize=10)

    axes[1].imshow(heatmap_2d, cmap=heatmap_cmap); axes[1].axis('off')
    axes[1].set_title(f'{technique_name} Heatmap', fontsize=10)

    axes[2].imshow(overlay_rgb); axes[2].axis('off')
    conf = float(probs_row[target_idx])
    axes[2].set_title(f'Overlay (conf={conf:.2f})', fontsize=10)

    colors = ['#16a34a' if c==target_idx else '#ef4444' for c in range(NUM_CLASSES)]
    axes[3].barh(CLASS_NAMES, probs_row, color=colors)
    axes[3].set_xlim(0,1); axes[3].set_title('Class Probabilities', fontsize=10)
    axes[3].tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.close()

def _reshape_transform_for(model):
    """Grad-CAM (and only Grad-CAM — Integrated Gradients and Occlusion work
    directly on input pixels and are unaffected) needs the target layer's
    raw output in (batch, channels, height, width) format. CNN backbones
    already produce that at their last stage, so no transform is needed
    (returning None tells pytorch_grad_cam to use the tensor as-is).
    Swin-Tiny's last stage outputs (batch, height, width, channels) —
    channels-last — so it just needs a permute. DINOv2-Base's last block
    outputs a flat (batch, num_prefix_tokens + num_patches, channels)
    token sequence (5 CLS/register prefix tokens here), which is not a
    spatial tensor at all until the prefix tokens are dropped and the
    remaining patch tokens are reshaped back into their 2D grid."""
    mk = model.model_key
    if mk == 'Swin-Tiny':
        return lambda t: t.permute(0,3,1,2)
    if mk == 'DINOv2-Base':
        inner = getattr(model.backbone, 'model', model.backbone)
        num_prefix = getattr(inner, 'num_prefix_tokens', 1)
        def _rt(t):
            b,n,c = t.shape
            patches = t[:, num_prefix:, :]
            side = int(round(patches.shape[1] ** 0.5))
            return patches.reshape(b, side, side, c).permute(0,3,1,2)
        return _rt
    return None  # CNN family: native NCHW, no transform needed


def save_combined_grid(mk, technique_name, mode, cases_data, out_path):
    """All disease classes present for this model/technique/mode, arranged
    in one image: each row is one class, columns are Original | Heatmap |
    Overlay | Class probabilities — the same four-panel layout as the
    individual per-case images, just stacked together for a quick side-by-
    side read across every disease at once."""
    class_ids = sorted(cases_data.keys())
    n = len(class_ids)
    if n == 0: return
    fig, axes = plt.subplots(n, 4, figsize=(20, 4.6*n))
    if n == 1: axes = axes.reshape(1,4)
    fig.suptitle(f'{technique_name} on {mode.upper()} Predictions — {mk} (all classes)',
                fontsize=15, fontweight='bold', y=1.002)
    for row, ci in enumerate(class_ids):
        image_tensor, true_idx, target_idx, probs_row, heat, overlay, cmap = cases_data[ci]
        orig = denorm(image_tensor).permute(1,2,0).cpu().numpy()
        axes[row,0].imshow(orig); axes[row,0].axis('off')
        title0 = f'True: {CLASS_NAMES[ci]}'
        if mode=='wrong': title0 += f' | Pred: {CLASS_NAMES[target_idx]}'
        axes[row,0].set_title(title0, fontsize=9)
        axes[row,1].imshow(heat, cmap=cmap); axes[row,1].axis('off')
        if row==0: axes[row,1].set_title(f'{technique_name} Heatmap', fontsize=9)
        axes[row,2].imshow(overlay); axes[row,2].axis('off')
        conf = float(probs_row[target_idx])
        axes[row,2].set_title(f'Overlay (conf={conf:.2f})', fontsize=9)
        colors = ['#16a34a' if c==target_idx else '#ef4444' for c in range(NUM_CLASSES)]
        axes[row,3].barh(CLASS_NAMES, probs_row, color=colors)
        axes[row,3].set_xlim(0,1); axes[row,3].tick_params(labelsize=6.5)
        if row==0: axes[row,3].set_title('Class Probabilities', fontsize=9)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()


# Accumulates every model's overlay image, keyed by technique -> mode ->
# model_key -> class_idx, so a single cross-model comparison grid (all
# completed backbones x all 8 disease classes) can be built once per
# technique per mode after every model has been processed.
CROSS_MODEL_XAI = {'gradcam':{'correct':{},'wrong':{}},
                   'integrated_gradients':{'correct':{},'wrong':{}},
                   'occlusion':{'correct':{},'wrong':{}}}

def save_cross_model_grid(technique_key, technique_name, mode, out_path):
    """One overlay per (model, disease class) cell, rows=models, columns=
    disease classes — lets every completed backbone be compared side by
    side for a given technique and correctness mode in a single image."""
    data = CROSS_MODEL_XAI[technique_key][mode]
    model_keys = [mk for mk in CFG['model_order'] if mk in data]
    if not model_keys: return
    fig, axes = plt.subplots(len(model_keys), NUM_CLASSES,
                             figsize=(2.3*NUM_CLASSES, 2.3*len(model_keys)))
    if len(model_keys)==1: axes = axes.reshape(1,NUM_CLASSES)
    fig.suptitle(f'{technique_name} — {mode.upper()} predictions — all models x all classes',
                fontsize=15, fontweight='bold', y=1.002)
    for row, mk in enumerate(model_keys):
        for col in range(NUM_CLASSES):
            ax = axes[row,col]
            if col in data[mk]:
                ax.imshow(data[mk][col])
            ax.axis('off')
            if row==0: ax.set_title(CLASS_NAMES[col][:12], fontsize=8)
            if col==0: ax.text(-0.15,0.5,mk,transform=ax.transAxes,fontsize=8,
                               ha='right',va='center',rotation=0,fontweight='bold')
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()


print('XAI shared infrastructure ready: wrapper, target-layer resolver, '
      'reshape_transform resolver (Swin/DINOv2-aware), four-panel writer, '
      'combined per-model grid writer, cross-model grid writer.')


XAI shared infrastructure ready: wrapper, target-layer resolver, reshape_transform resolver (Swin/DINOv2-aware), four-panel writer, combined per-model grid writer, cross-model grid writer.


In [37]:
def gradcam_case(model, image_tensor, target_idx):
    wrapped = FineHeadWrapper(model).to(DEVICE).eval()
    cam = GradCAMPlusPlus(model=wrapped, target_layers=_target_layer(model),
                          reshape_transform=_reshape_transform_for(model))
    inp = image_tensor.unsqueeze(0).to(DEVICE).clone().requires_grad_(True)
    heat = cam(input_tensor=inp, targets=[ClassifierOutputTarget(target_idx)])[0]
    rgb = denorm(image_tensor).permute(1,2,0).cpu().numpy()
    overlay = show_cam_on_image(rgb, heat, use_rgb=True)
    return heat, overlay

print('Grad-CAM++ for every completed model, both prediction modes, one image per disease class, '
      'plus a combined per-model grid and cross-model comparison accumulation...')
for mk, model in MODELS.items():
    xai_dir = model_dirs(mk)['xai']/'gradcam'
    xai_dir.mkdir(exist_ok=True)
    probs_all = RESULTS_THRESHOLDED[mk]['probs']
    lbl_all   = RESULTS_THRESHOLDED[mk]['lbl']
    pred_all  = RESULTS_THRESHOLDED[mk]['pred']
    for mode in ('correct','wrong'):
        cases = pick_one_case_per_class(mk, mode)
        combined_data = {}
        for ci, idx in cases.items():
            image_tensor, true_lbl = test_ds[idx]
            target_idx = ci if mode=='correct' else int(pred_all[idx])
            heat, overlay = gradcam_case(model, image_tensor, target_idx)
            out_path = xai_dir/f'gradcam_{mode}_{model_key_safe(mk)}_{CLASS_NAMES[ci].replace(" ","_")}.png'
            save_xai_panel(mk,'Grad-CAM++',mode,CLASS_NAMES[ci],image_tensor,
                          true_lbl,target_idx,probs_all[idx],heat,overlay,
                          out_path,heatmap_cmap='jet')
            combined_data[ci] = (image_tensor, true_lbl, target_idx, probs_all[idx], heat, overlay, 'jet')
            CROSS_MODEL_XAI['gradcam'][mode].setdefault(mk, {})[ci] = overlay
        combined_path = xai_dir/f'gradcam_{mode}_{model_key_safe(mk)}_ALL_CLASSES.png'
        save_combined_grid(mk,'Grad-CAM++',mode,combined_data,combined_path)
    print(f'  {mk}: done.')


Grad-CAM++ for every completed model, both prediction modes, one image per disease class, plus a combined per-model grid and cross-model comparison accumulation...
  ConvNeXt-S: done.
  ConvNeXt-Base: done.
  ConvFormer-S18: done.
  DenseNet121: done.
  InceptionNeXt-S: done.
  HGNetV2-B4: done.
  EfficientNetV2-S: done.
  Swin-Tiny: done.
  DINOv2-Base: done.


In [38]:
def integrated_gradients_case(model, image_tensor, target_idx, n_steps=24, internal_batch_size=4):
    wrapped = FineHeadWrapper(model).to(DEVICE).eval()
    ig = IntegratedGradients(wrapped)
    inp = image_tensor.unsqueeze(0).to(DEVICE)
    attr = ig.attribute(inp, target=target_idx, n_steps=n_steps,
                        internal_batch_size=internal_batch_size)
    heat = attr.squeeze(0).abs().sum(0).detach().cpu().numpy()
    heat = _normalize_map(heat)
    rgb = denorm(image_tensor).permute(1,2,0).cpu().numpy()
    overlay = show_cam_on_image(rgb, heat, use_rgb=True)
    return heat, overlay

print('Integrated Gradients for every completed model, both prediction modes, one image per disease class, '
      'plus a combined per-model grid and cross-model comparison accumulation...')
for mk, model in MODELS.items():
    xai_dir = model_dirs(mk)['xai']/'integrated_gradients'
    xai_dir.mkdir(exist_ok=True)
    probs_all = RESULTS_THRESHOLDED[mk]['probs']
    pred_all  = RESULTS_THRESHOLDED[mk]['pred']
    for mode in ('correct','wrong'):
        cases = pick_one_case_per_class(mk, mode)
        combined_data = {}
        for ci, idx in cases.items():
            image_tensor, true_lbl = test_ds[idx]
            target_idx = ci if mode=='correct' else int(pred_all[idx])
            try:
                heat, overlay = integrated_gradients_case(model, image_tensor, target_idx)
            except Exception as ex:
                print(f'    [WARN] IG failed for {mk} {mode} {CLASS_NAMES[ci]}: {ex}')
                continue
            out_path = xai_dir/f'ig_{mode}_{model_key_safe(mk)}_{CLASS_NAMES[ci].replace(" ","_")}.png'
            save_xai_panel(mk,'Integrated Gradients',mode,CLASS_NAMES[ci],image_tensor,
                          true_lbl,target_idx,probs_all[idx],heat,overlay,
                          out_path,heatmap_cmap='inferno')
            combined_data[ci] = (image_tensor, true_lbl, target_idx, probs_all[idx], heat, overlay, 'inferno')
            CROSS_MODEL_XAI['integrated_gradients'][mode].setdefault(mk, {})[ci] = overlay
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
        combined_path = xai_dir/f'ig_{mode}_{model_key_safe(mk)}_ALL_CLASSES.png'
        save_combined_grid(mk,'Integrated Gradients',mode,combined_data,combined_path)
    print(f'  {mk}: done.')


Integrated Gradients for every completed model, both prediction modes, one image per disease class, plus a combined per-model grid and cross-model comparison accumulation...
  ConvNeXt-S: done.
  ConvNeXt-Base: done.
  ConvFormer-S18: done.
  DenseNet121: done.
  InceptionNeXt-S: done.
  HGNetV2-B4: done.
  EfficientNetV2-S: done.
  Swin-Tiny: done.
  DINOv2-Base: done.


In [39]:
# Sliding-window settings chosen as a balance between spatial resolution and
# runtime: a 48x48 window with 32-pixel strides on a 224x224 image is ~49
# forward passes per case. This is the slowest of the three XAI techniques —
# it runs once after training finishes, not during training.
OCCLUSION_WINDOW  = (3,48,48)
OCCLUSION_STRIDES = (3,32,32)

def occlusion_case(model, image_tensor, target_idx):
    wrapped = FineHeadWrapper(model).to(DEVICE).eval()
    occ = Occlusion(wrapped)
    inp = image_tensor.unsqueeze(0).to(DEVICE)
    attr = occ.attribute(inp, target=target_idx, strides=OCCLUSION_STRIDES,
                         sliding_window_shapes=OCCLUSION_WINDOW, baselines=0)
    heat = attr.squeeze(0).abs().sum(0).detach().cpu().numpy()
    heat = _normalize_map(heat)
    rgb = denorm(image_tensor).permute(1,2,0).cpu().numpy()
    overlay = show_cam_on_image(rgb, heat, use_rgb=True)
    return heat, overlay

print('Occlusion sensitivity for every completed model, both prediction modes, one image per disease class, '
      'plus a combined per-model grid and cross-model comparison accumulation...')
for mk, model in MODELS.items():
    xai_dir = model_dirs(mk)['xai']/'occlusion'
    xai_dir.mkdir(exist_ok=True)
    probs_all = RESULTS_THRESHOLDED[mk]['probs']
    pred_all  = RESULTS_THRESHOLDED[mk]['pred']
    for mode in ('correct','wrong'):
        cases = pick_one_case_per_class(mk, mode)
        combined_data = {}
        for ci, idx in cases.items():
            image_tensor, true_lbl = test_ds[idx]
            target_idx = ci if mode=='correct' else int(pred_all[idx])
            try:
                heat, overlay = occlusion_case(model, image_tensor, target_idx)
            except Exception as ex:
                print(f'    [WARN] Occlusion failed for {mk} {mode} {CLASS_NAMES[ci]}: {ex}')
                continue
            out_path = xai_dir/f'occlusion_{mode}_{model_key_safe(mk)}_{CLASS_NAMES[ci].replace(" ","_")}.png'
            save_xai_panel(mk,'Occlusion',mode,CLASS_NAMES[ci],image_tensor,
                          true_lbl,target_idx,probs_all[idx],heat,overlay,
                          out_path,heatmap_cmap='viridis')
            combined_data[ci] = (image_tensor, true_lbl, target_idx, probs_all[idx], heat, overlay, 'viridis')
            CROSS_MODEL_XAI['occlusion'][mode].setdefault(mk, {})[ci] = overlay
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
        combined_path = xai_dir/f'occlusion_{mode}_{model_key_safe(mk)}_ALL_CLASSES.png'
        save_combined_grid(mk,'Occlusion',mode,combined_data,combined_path)
    print(f'  {mk}: done.')


Occlusion sensitivity for every completed model, both prediction modes, one image per disease class, plus a combined per-model grid and cross-model comparison accumulation...
  ConvNeXt-S: done.
  ConvNeXt-Base: done.
  ConvFormer-S18: done.
  DenseNet121: done.
  InceptionNeXt-S: done.
  HGNetV2-B4: done.
  EfficientNetV2-S: done.
  Swin-Tiny: done.
  DINOv2-Base: done.


In [40]:
print('Building cross-model XAI comparison grids (all completed backbones x all 8 disease classes)...')
technique_labels = {'gradcam':'Grad-CAM++','integrated_gradients':'Integrated Gradients','occlusion':'Occlusion'}
for tech_key, tech_label in technique_labels.items():
    for mode in ('correct','wrong'):
        out_path = SHARED_DIRS['comparison']/f'xai_cross_model_{tech_key}_{mode}.png'
        save_cross_model_grid(tech_key, tech_label, mode, out_path)
        n_models = len(CROSS_MODEL_XAI[tech_key][mode])
        print(f'  {tech_label} ({mode}): {n_models} model(s) -> {out_path}')


Building cross-model XAI comparison grids (all completed backbones x all 8 disease classes)...
  Grad-CAM++ (correct): 9 model(s) -> results_v12_9backbone_noKD_RGB\comparison\xai_cross_model_gradcam_correct.png
  Grad-CAM++ (wrong): 9 model(s) -> results_v12_9backbone_noKD_RGB\comparison\xai_cross_model_gradcam_wrong.png
  Integrated Gradients (correct): 9 model(s) -> results_v12_9backbone_noKD_RGB\comparison\xai_cross_model_integrated_gradients_correct.png
  Integrated Gradients (wrong): 9 model(s) -> results_v12_9backbone_noKD_RGB\comparison\xai_cross_model_integrated_gradients_wrong.png
  Occlusion (correct): 9 model(s) -> results_v12_9backbone_noKD_RGB\comparison\xai_cross_model_occlusion_correct.png
  Occlusion (wrong): 9 model(s) -> results_v12_9backbone_noKD_RGB\comparison\xai_cross_model_occlusion_wrong.png


In [41]:
print(f'Best model by TUNE macro recall: {WINNER_KEY}\n')
winner_xai_dir = model_dirs(WINNER_KEY)['xai']
techniques = ['gradcam','integrated_gradients','occlusion']
technique_labels = {'gradcam':'Grad-CAM++','integrated_gradients':'Integrated Gradients','occlusion':'Occlusion'}

for tech in techniques:
    tech_dir = winner_xai_dir/tech
    files = sorted(tech_dir.glob('*.png')) if tech_dir.exists() else []
    print(f'{technique_labels[tech]}: {len(files)} images -> {tech_dir}')

print(f'\nDisplaying every {WINNER_KEY} XAI panel inline (loaded from the saved images above)...')
for tech in techniques:
    tech_dir = winner_xai_dir/tech
    files = sorted(tech_dir.glob('*.png')) if tech_dir.exists() else []
    for f in files:
        img = Image.open(f)
        fig, ax = plt.subplots(figsize=(11,2.5))
        ax.imshow(img); ax.axis('off')
        plt.tight_layout()
        plt.show()
        plt.close(fig)


Best model by TUNE macro recall: Swin-Tiny

Grad-CAM++: 16 images -> results_v12_9backbone_noKD_RGB\models\Swin_Tiny\xai\gradcam
Integrated Gradients: 16 images -> results_v12_9backbone_noKD_RGB\models\Swin_Tiny\xai\integrated_gradients
Occlusion: 16 images -> results_v12_9backbone_noKD_RGB\models\Swin_Tiny\xai\occlusion

Displaying every Swin-Tiny XAI panel inline (loaded from the saved images above)...


In [42]:
@torch.no_grad()
def save_inference_bundle(model,dataset,mk,split):
    bs=get_model_cfg(mk)['batch_size']
    ld=DataLoader(dataset,batch_size=bs,shuffle=False,num_workers=0)
    model.eval()
    af,ar,ac,al=[],[],[],[]
    for imgs,labels in ld:
        imgs=imgs.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
            fine,risk,cpn=model(imgs)
        af.append(fine.float().cpu().numpy())
        ar.append(risk.float().cpu().numpy())
        ac.append(cpn.float().cpu().numpy())
        al.append(labels.numpy())
    paths=np.array([str(p) for p,_ in dataset.samples])
    fname=SHARED_DIRS['ensemble_bundles']/f'inference_{model_key_safe(mk)}_{split}.npz'
    np.savez(fname,
             fine_logits=np.concatenate(af),
             risk_logits=np.concatenate(ar),
             cpn_logits =np.concatenate(ac),
             labels     =np.concatenate(al),
             paths      =paths,
             class_names=np.array(CLASS_NAMES),
             risk_names =np.array(RISK_NAMES),
             cpn_classes=np.array(CPN_CLASSES))
    return fname

print('Saving per-model inference outputs (val + test)...')
for mk,model in MODELS.items():
    fv=save_inference_bundle(model,val_ds, mk,'val')
    ft=save_inference_bundle(model,test_ds,mk,'test')
    print(f'  {mk:<18} -> {fv.name} | {ft.name}')
print(f"Saved -> {SHARED_DIRS['ensemble_bundles']}")

for mk,model in MODELS.items():
    bundle={
        'model_state'        :model.state_dict(),
        'model_key'          :mk,
        'use_cbam'           :model.use_cbam,
        'feature_dim'        :model.feature_dim,
        'class_names'        :CLASS_NAMES,'class_to_idx':CLASS_TO_IDX,
        'num_classes'        :NUM_CLASSES,'risk_names':RISK_NAMES,'cpn_classes':CPN_CLASSES,
        'img_size'           :CFG['img_size'],'mean':CFG['mean'],'std':CFG['std'],
        'arcface_m'          :CFG['arcface_m'],
        'thresholds'         :THRESHOLDS[mk],
        'test_metrics'       :RESULTS[mk]['met'],
        'test_metrics_thresh':RESULTS_THRESHOLDED[mk]['met'],
        'risk_disease_flag'  :{'precision':RISK_FLAGS[mk]['precision'],
                               'recall':RISK_FLAGS[mk]['recall'],
                               'n_flagged':RISK_FLAGS[mk]['n_flagged']},
        'is_best'            :(mk==WINNER_KEY),
        'effective_config'   :get_model_cfg(mk),
        'version'            :'v12-9backbone-noKD-custom_lora-arcface-hierarchical-mcc-curriculum-riskconsistency-cpnlink',
    }
    fname=model_dirs(mk)['bundle']/'model_bundle.pth'
    torch.save(bundle,fname); print(f'  {fname}')
print('\nAll per-model bundles saved.')


Saving per-model inference outputs (val + test)...
  ConvNeXt-S         -> inference_ConvNeXt_S_val.npz | inference_ConvNeXt_S_test.npz
  ConvNeXt-Base      -> inference_ConvNeXt_Base_val.npz | inference_ConvNeXt_Base_test.npz
  ConvFormer-S18     -> inference_ConvFormer_S18_val.npz | inference_ConvFormer_S18_test.npz
  DenseNet121        -> inference_DenseNet121_val.npz | inference_DenseNet121_test.npz
  InceptionNeXt-S    -> inference_InceptionNeXt_S_val.npz | inference_InceptionNeXt_S_test.npz
  HGNetV2-B4         -> inference_HGNetV2_B4_val.npz | inference_HGNetV2_B4_test.npz
  EfficientNetV2-S   -> inference_EfficientNetV2_S_val.npz | inference_EfficientNetV2_S_test.npz
  Swin-Tiny          -> inference_Swin_Tiny_val.npz | inference_Swin_Tiny_test.npz
  DINOv2-Base        -> inference_DINOv2_Base_val.npz | inference_DINOv2_Base_test.npz
Saved -> results_v12_9backbone_noKD_RGB\ensemble_bundles
  results_v12_9backbone_noKD_RGB\models\ConvNeXt_S\bundle\model_bundle.pth
  results_v12_

In [43]:
print('\n'+'='*72)
print(f'V12 Pipeline complete — {len(MODELS)}/{len(CFG["model_order"])} backbones trained')
print('='*72)
for mk in MODELS:
    m=RESULTS[mk]['met']; mt=RESULTS_THRESHOLDED[mk]['met']; rf=RISK_FLAGS[mk]
    tag='  *** BEST ***' if mk==WINNER_KEY else ''
    print(f'\n  {mk}{tag}')
    print(f'    RAW  F1={m["f1_macro"]:.4f} Acc={m["accuracy"]:.4f} '
          f'AUC={m.get("auc_macro",float("nan")):.4f} Rec={m["rec_macro"]:.4f}')
    print(f'    TUNE F1={mt["f1_macro"]:.4f} Acc={mt["accuracy"]:.4f} '
          f'AUC={mt.get("auc_macro",float("nan")):.4f} Rec={mt["rec_macro"]:.4f}')
    print(f'    Risk/Disease flag: {rf["n_flagged"]} flagged | '
          f'precision={rf["precision"]:.3f} | recall={rf["recall"]:.3f}')
print(f'\nOutput directory: {CFG["out_dir"].resolve()}')



V12 Pipeline complete — 9/9 backbones trained

  ConvNeXt-S
    RAW  F1=0.8022 Acc=0.8091 AUC=0.9698 Rec=0.8257
    TUNE F1=0.7978 Acc=0.8136 AUC=0.9698 Rec=0.8111
    Risk/Disease flag: 21 flagged | precision=0.476 | recall=0.244

  ConvNeXt-Base
    RAW  F1=0.8282 Acc=0.8318 AUC=0.9569 Rec=0.8267
    TUNE F1=0.8240 Acc=0.8273 AUC=0.9569 Rec=0.8240
    Risk/Disease flag: 5 flagged | precision=0.600 | recall=0.079

  ConvFormer-S18
    RAW  F1=0.7842 Acc=0.7955 AUC=0.9726 Rec=0.7884
    TUNE F1=0.7698 Acc=0.7818 AUC=0.9726 Rec=0.7728
    Risk/Disease flag: 6 flagged | precision=0.500 | recall=0.062

  DenseNet121
    RAW  F1=0.7358 Acc=0.7455 AUC=0.9601 Rec=0.7599
    TUNE F1=0.7395 Acc=0.7545 AUC=0.9601 Rec=0.7603
    Risk/Disease flag: 7 flagged | precision=0.429 | recall=0.056

  InceptionNeXt-S
    RAW  F1=0.7375 Acc=0.7636 AUC=0.9610 Rec=0.7604
    TUNE F1=0.7303 Acc=0.7455 AUC=0.9610 Rec=0.7498
    Risk/Disease flag: 30 flagged | precision=0.567 | recall=0.304

  HGNetV2-B4
    